# Function

In [ ]:
!pip install supabase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.8 MB/s eta 0:00:00


In [ ]:
import argparse
import os
import re

import numpy as np
import math
import pandas as pd
from supabase import create_client
from postgrest import APIError

# IDX

In [ ]:
# JISDOR
jisdor = {
    '2022': 15592,
    '2023': 15439,
    '2024': 16157,
    '2025': 16785
}

## 2025 FY version (Coal Companies Breakdown + Cost of Revenue Breakdown)

In [ ]:
def _get_sheet_names(excel_file_path):
    excel_file = pd.ExcelFile(excel_file_path)
    sheet_names = excel_file.sheet_names
    # sheet_names = ['ITMG']
    return sheet_names


def _filter_others_keys(row: pd.Series):
    """
    Removes keys containing \"Other\" by replacing their value with NaN,
    and return its values as series to be summed

    MODIFIES the DataFrame directly!
    """
    if re.search("^[Oo]ther", row.name):
        value = row.value
        row.value = np.NaN
        return value

def none_value_extractor(metrics):
    if pd.isna(metrics):
        return None
    else:
        return metrics

def _reduce_others_keys(df: pd.DataFrame):
    """
    Groups all \"Other\" breakdown entries into a single \"Others\" by summing them up

    MODIFIES the DataFrame directly!
    """

    others_rev = df[df.index.str.contains("^[Oo]ther")]
    df = df[~df.index.str.contains("^[Oo]ther")]

    if others_rev.shape[0] > 0:
        df.loc["Others", "value"] = int(others_rev["value"].sum())
        df.loc["Others", "category"] = others_rev['category'].iloc[0]
        return df
    else:
        return df

def _process_sheet(excel_file_path, sheet_name, usd_idr_rate, bank_company):
    df = pd.read_excel(excel_file_path, sheet_name=sheet_name, header=None)

    data = df[[0,1]]
    data = data.rename(columns={0:'key', 1:'value'})
    data = data.dropna(subset=['key'])
    data = data.set_index('key')
    metadata = data.loc['symbol':'currency'].copy()
    # replace NaN value with None for postgrest json
    metadata = metadata.replace({np.nan: None})

    currency = metadata.loc['currency'].value

    if bank_company:

        # Balance sheet
        bal_sheet = df.iloc[:,13:15]
        bal_sheet.columns = ["key",'value']
        bal_sheet['group'] = bal_sheet['key'].where(bal_sheet['value'].isna()).ffill()
        bal_sheet = bal_sheet.set_index("key")

        # Non-loan-asset breakdown
        non_loan_asset_bd = df[[15,16,17]]
        non_loan_asset_bd = non_loan_asset_bd.rename(columns={15:'class',16:'category', 17:'amount'}).dropna(subset=['category'])
        non_loan_asset_bd = non_loan_asset_bd.set_index('category')
        non_loan_asset_bd = non_loan_asset_bd.drop(index=["sum", "match",'Breakdown of: Non-loan asset']).reset_index()

        balance_sheet = {
            "gross_loan": none_value_extractor(bal_sheet.loc['Gross Loan'].value),
            "allowance_for_loans": none_value_extractor(bal_sheet.loc['Allowance for Loans'].value),
            "net_loan": none_value_extractor(bal_sheet.loc['NET LOAN'].value),
            'earning_asset': none_value_extractor(non_loan_asset_bd[non_loan_asset_bd['class'] == "Earning Asset"].amount.sum() + bal_sheet.loc['NET LOAN'].value if pd.isna(bal_sheet.loc['Total Earning Asset'].value) else bal_sheet.loc['Total Earning Asset'].value),
            "non_loan_asset": none_value_extractor(bal_sheet.loc['Non-Loan Asset'].value),
            "total_asset": none_value_extractor(bal_sheet.loc['TOTAL ASSET'].value),
            "current_account": none_value_extractor(bal_sheet.loc['Current Account'].value),
            "savings_account": none_value_extractor(bal_sheet.loc['Savings Account'].value),
            "time_deposit": none_value_extractor(bal_sheet.loc['Time Deposits'].value),
            "total_deposit": none_value_extractor(bal_sheet.loc['TOTAL DEPOSIT'].value),
            "other_interest_bearing_liabilities": none_value_extractor(bal_sheet.loc['Other Interest-Bearing Liabilities'].value),
            "non_interest_bearing_liabilities": none_value_extractor(bal_sheet.loc['Non-Interest-Bearning Liabilities'].value),
            "total_liabilities": none_value_extractor(bal_sheet.loc['TOTAL LIABILITIES'].value),
            "total_equity": none_value_extractor(bal_sheet.loc["TOTAL SHAREHOLDER'S EQUITY"].value),
            "core_capital_tier1": none_value_extractor(bal_sheet.loc['Core Capital (Tier 1)'].value),
            "supplementary_capital_tier2": none_value_extractor(bal_sheet.loc['Supplementary Capital (Tier 2)'].value),
            "total_capital": none_value_extractor(bal_sheet.loc['TOTAL CAPITAL'].value),
            "credit_rwa": none_value_extractor(bal_sheet.loc['RWAs Considering Credit Risk'].value),
            "market_rwa": none_value_extractor(bal_sheet.loc['RWAs Considering Market Risk'].value),
            "operational_rwa": none_value_extractor(bal_sheet.loc['RWAs Considering Operational Risk'].value),
            "total_risk_weighted_asset": none_value_extractor(bal_sheet.loc['TOTAL RISK-WEIGHTED ASSETS'].value)
            }

        for metric in list(balance_sheet.keys()):
            try:
                balance_sheet[metric] = int(balance_sheet[metric])
                if currency == "USD":
                    balance_sheet[metric] = int(balance_sheet[metric] * usd_idr_rate)
            except:
                pass

        # Income Statement
        income_stmt = data.loc['interest income':'net income'].copy()
        income_stmt['value'] = income_stmt['value'].fillna(0)

        int_income_bd = df[[2,3,4]]
        int_income_bd = int_income_bd.rename(columns={2:'category',3:'key', 4:'value'}).dropna(subset=['key'])
        int_income_bd = int_income_bd.set_index('key')
        int_income_bd = int_income_bd.drop(index=["sum", "match",'subsector','Breakdown of: total revenue','Year'])
        int_income_bd = int_income_bd.dropna(subset=['value'])
        # Sum up all "Other" expenses into a single entry of "Others"
        int_income_bd = _reduce_others_keys(int_income_bd)
        int_income_bd = int_income_bd.reset_index()

        expense_bd = df[[5,6,7]]
        expense_bd = expense_bd.rename(columns={5:'category',6:'key', 7:'value'}).dropna(subset=['key'])
        expense_bd = expense_bd.set_index('key')
        expense_bd = expense_bd.drop(index=["sum", "match",'Breakdown of: operating expenses'])
        # check for positive expense; expense is standardized as having negative values only
        if (expense_bd['value'].values > 0).any():
            raise ValueError(f'Positive value detected in expense breakdown of sheet {sheet_name}')
        # Sum up all "Other" expenses into a single entry of "Others"
        expense_bd = _reduce_others_keys(expense_bd)
        # rename the expense index if any match is found in income breakdown
        for expense_idx in expense_bd.index:
            if expense_idx in int_income_bd['key'].values:
                expense_bd.rename(index={expense_idx: expense_idx + ' (expense)'}, inplace=True)
            elif expense_idx == "Others":
                expense_bd.rename(index={expense_idx: expense_idx + ' (expense)'}, inplace=True)

        expense_bd = expense_bd.reset_index()

        try:
            non_operating_income = income_stmt.loc['net non operating income/(expenses)'].value
        except:
            non_operating_income = income_stmt.loc['net non operating income'].value

        input = {
            "symbol": metadata.loc['symbol'].value,
            "url": metadata.loc['url'].value,
            "interest_income": none_value_extractor(income_stmt.loc['interest income'].value),
            "interest_expense": none_value_extractor(-(income_stmt.loc['interest expenses'].value)),
            "net_interest_income": none_value_extractor(income_stmt.loc['net interest income'].value),
            "premium_income": none_value_extractor(income_stmt.loc['premium income'].value),
            "premium_expense": none_value_extractor(-(income_stmt.loc['premium expense'].value)),
            "net_premium_income": none_value_extractor(income_stmt.loc['net premium income'].value),
            "non_interest_income": none_value_extractor(income_stmt.loc['non interest income'].value),
            "total_revenue": none_value_extractor(income_stmt.loc['total revenue'].value),
            "operating_expense": none_value_extractor(-(income_stmt.loc['operating expenses'].value)),
            "provision": none_value_extractor(-(income_stmt.loc['provision for impairment'].value)),
            "operating_income": none_value_extractor(income_stmt.loc['net operating income'].value),
            "non_operating_income_or_loss": none_value_extractor(non_operating_income),
            "pretax_income": none_value_extractor(income_stmt.loc['pretax income'].value),
            "income_taxes": none_value_extractor(-(income_stmt.loc['tax'].value)),
            "net_income": none_value_extractor(income_stmt.loc['net income'].value),
            "diluted_shares_outstanding": none_value_extractor(bal_sheet.loc['Weighted average shares outstanding'].value if bal_sheet.loc['Weighted average shares outstanding'].value != 0 else None),
            "int_income_breakdown": [{"class":int_income_bd.iloc[i,1],"category": int_income_bd.iloc[i,0], "amount": (int_income_bd.iloc[i,2])} for i in range (int_income_bd.shape[0])],
            "operating_expense_breakdown": [{"class":expense_bd.iloc[i,1],"category": expense_bd.iloc[i,0], "amount": int(-(expense_bd.iloc[i,2]))} for i in range (expense_bd.shape[0])]
            }

        for metric in ['interest_income', 'interest_expense', 'net_interest_income', 'premium_income', 'premium_expense', 'net_premium_income', 'non_interest_income', 'total_revenue', 'operating_expense', 'provision', 'operating_income',"non_operating_income_or_loss", 'pretax_income', 'income_taxes', 'net_income']:
            input[metric] = int(input[metric])
            if currency == "USD":
                input[metric] = int(input[metric] * usd_idr_rate)

        for breakdown in ['int_income_breakdown', 'operating_expense_breakdown']:
            for item in input[breakdown]:
                item['amount'] = int(item['amount'])
                if currency == "USD":
                    item['amount'] = int(item['amount'] * usd_idr_rate)

        # Industry Breakdown
        # try:
        #     sector_loan_breakdown = bal_sheet[bal_sheet.group == "Breakdown of: gross loan (by economic sector)"]
        #     sector_loan_breakdown = sector_loan_breakdown.drop(index=["sum", 'Breakdown of: gross loan (by economic sector)'])
        #     sector_loan_breakdown = sector_loan_breakdown.dropna(subset=['value'])
        #     sector_loan_breakdown['group'] = "gross_loan"
        # except:
        sector_loan_breakdown = bal_sheet[bal_sheet.group == "Breakdown of: loan receivable (by economic sector)"]
        sector_loan_breakdown = sector_loan_breakdown.drop(index=["sum", 'Breakdown of: loan receivable (by economic sector)'])
        sector_loan_breakdown = sector_loan_breakdown.dropna(subset=['value'])
        sector_loan_breakdown['group'] = "loan_receivable"

        try:
            special_mention_loan = bal_sheet.loc['Special Mention Loan (Loan receivable)'].value
        except:
            special_mention_loan = bal_sheet.loc['Special Mention Loan'].value

        try:
            restructured_loan = bal_sheet.loc['Restructured Loan (current)'].value
        except:
            restructured_loan = bal_sheet.loc['Restructured Loan'].value

        industry_breakdown = {
            "loan_at_risk":{
            "Special Mention Loan": special_mention_loan,
            "Restructured Loan (current)": restructured_loan,
            "Non-performing Loan (NPL)": bal_sheet.loc['Non-performing Loan (NPL)'].value
            },
            "loan_by_economic_sectors": sector_loan_breakdown.reset_index().to_dict(orient='records'),
            "non_loan_asset": non_loan_asset_bd.to_dict(orient='records')
        }

        # Cash Flow
        cash_fl = df[[19,20]]#.dropna()
        cash_fl.columns = ["key",'value']
        cash_fl = cash_fl.set_index("key")

        cash_flow = {
                    "high_quality_liquid_asset": none_value_extractor(cash_fl.loc['Total High Quality Liquid Asset (HQLA)'].value),
                    "cash_outflow": none_value_extractor(cash_fl.loc['Cash Outflow'].value),
                    "cash_inflow": none_value_extractor(cash_fl.loc['Cash Inflow'].value),
                    "end_cash_position": none_value_extractor(cash_fl.loc['TOTAL NET CASH OUTFLOWS'].value),
                    "operating_cash_flow": cash_fl.loc['Cash Flows from Operating Activities'].value,
                    "investing_cash_flow": cash_fl.loc['Cash Flows from Investing Activities'].value,
                    "financing_cash_flow": cash_fl.loc['Cash Flows from Financing Activities'].value,
                    "net_cash_flow": cash_fl.loc['NET INCREASE/DECREASED'].value,
                    "realized_capital_goods_investment": none_value_extractor(cash_fl.loc['REALIZED CAPITAL GOODS INVESTMENT'].value),
                    "free_cash_flow": none_value_extractor(cash_fl.loc['Cash Flows from Operating Activities'].value - cash_fl.loc['REALIZED CAPITAL GOODS INVESTMENT'].value),
                    }

        for metric in list(cash_flow.keys()):
            if currency == "USD":
                try:
                    cash_flow[metric] = int(cash_flow[metric])
                    cash_flow[metric] = int(cash_flow[metric] * usd_idr_rate)
                except:
                    pass

        # Employee
        employee = df.iloc[:,22:24]
        employee.columns = ['key','value']

        employee = employee.dropna(subset="key").iloc[1:,:].set_index("key")

        employee.index = ["permanent_employee","contract_employee","others_employee","total_employee"]

        if employee.loc['total_employee'].value == 0:
            employee = None
        else:
            employee.value = employee.value.astype('Int64')
            employee = employee["value"].to_dict()

    else:  # NOT bank company
        # Income Statement Metrics
        income_stmt = data.loc['total revenue':'ebitda'].copy()
        income_stmt['value'] = income_stmt['value'].fillna(0)

        revenue_bd = df[[2,3,4]]
        revenue_bd = revenue_bd.rename(columns={2:'category',3:'key', 4:'value'}).dropna(subset=['key'])
        revenue_bd = revenue_bd.set_index('key')
        revenue_bd = revenue_bd.drop(index=["sum", "match",'subsector','Breakdown of: total revenue','Year'])

        # ## Sum up all "Other" expenses into a single entry of "Others"
        revenue_bd =_reduce_others_keys(revenue_bd)
        revenue_bd.reset_index(inplace=True)

        # Separate data before and after "Breakdown of: cost of revenue"
        cost_revenue_idx = revenue_bd[revenue_bd['key'] == 'Breakdown of: cost of revenue'].index
        if len(cost_revenue_idx) > 0:
            idx = cost_revenue_idx[0]
            cost_revenue_bd = revenue_bd.iloc[idx+1:].copy()

            cost_of_revenue_breakdown = cost_revenue_bd.groupby('category').apply(
                lambda group: group.set_index('key')['value'].to_dict()
            ).to_dict()
            revenue_bd = revenue_bd.iloc[:idx].copy()

        # Balance sheet Metrics
        bal_sheet = df.iloc[:50,13:15]
        bal_sheet.columns = ["key",'value']

        bal_sheet = bal_sheet.set_index("key")

        # Cash Flow Metrics
        cash_fl = df.iloc[:,16:18]
        cash_fl.columns = ["key",'value']

        cash_fl = cash_fl.dropna(subset=['key'])

        cash_fl = cash_fl.set_index("key")

        try:
            expense_bd = df[[5,6,7,8]]
            expense_bd = expense_bd.rename(columns={5:'category',6:'key', 7:'value', 8:'group'})

            breakdown_idx = expense_bd[expense_bd['key'] == 'OGC KPI'].index
            expense_bd = expense_bd.iloc[:breakdown_idx[0]]

            expense_bd = expense_bd[~expense_bd.key.isin(["sum","match",'Breakdown of: operating expenses'])]

            expense_bd["group"] = expense_bd["group"].ffill()

            expense_bd = expense_bd.dropna()

            expense_bd["key"] = expense_bd["group"] + '-' + expense_bd["key"]

            expense_bd = expense_bd.drop(['group'], axis=1)
        except:
            expense_bd = df[[5,6,7]]
            expense_bd = expense_bd.rename(columns={5:'category',6:'key', 7:'value'}).dropna(subset=['key']).reset_index(drop=True)

            breakdown_idx = expense_bd[expense_bd['key'] == 'OGC KPI'].index
            expense_bd = expense_bd.iloc[:breakdown_idx[0]]

            expense_bd = expense_bd.set_index('key')
            expense_bd = expense_bd.drop(index=["sum", "match",'Breakdown of: operating expenses'])
            # check for positive expense; expense is standardized as having negative values only
            if (expense_bd['value'].values > 0).any():
                raise ValueError(f'Positive value detected in expense breakdown of sheet {sheet_name}')
            # Sum up all "Other" expenses into a single entry of "Others"
            expense_bd = _reduce_others_keys(expense_bd)
            # rename the expense index if any match is found in income breakdown
            for expense_idx in expense_bd.index:
                if expense_idx in revenue_bd['key'].values:
                    expense_bd.rename(index={expense_idx: expense_idx + ' (expense)'}, inplace=True)
                elif expense_idx == "Others":
                    expense_bd.rename(index={expense_idx: expense_idx + ' (expense)'}, inplace=True)

            expense_bd.reset_index(inplace=True)

            expense_bd["value"] = expense_bd["value"].astype('int')

        input = {
            "symbol": metadata.loc['symbol'].value,
            "url": metadata.loc['url'].value,
            "total_revenue": none_value_extractor(income_stmt.loc['total revenue'].value),
            "cost_of_revenue": none_value_extractor(-(income_stmt.loc['cost of revenue'].value)),
            "gross_income": none_value_extractor(income_stmt.loc['gross income'].value),
            "operating_expense": none_value_extractor(-(income_stmt.loc['operating expenses'].value)),
            "operating_income": none_value_extractor(income_stmt.loc['net operating income'].value),
            "non_operating_income_or_loss": none_value_extractor(income_stmt.loc['net non operating income/(expenses)'].value),
            "pretax_income": none_value_extractor(income_stmt.loc['pretax income'].value),
            "income_taxes":  none_value_extractor(-(income_stmt.loc['tax'].value)),
            "net_income": none_value_extractor(income_stmt.loc['net income'].value),
            "interest_expense_non_operating": none_value_extractor(-(income_stmt.loc['non operating interest expense'].value)),
            "ebit": none_value_extractor(income_stmt.loc['ebit'].value),
            "ebitda": none_value_extractor(income_stmt.loc['ebit'].value + income_stmt.loc['depreciation and amortization'].value if pd.isna(income_stmt.loc['ebitda'].value) else income_stmt.loc['ebitda'].value),
            "diluted_shares_outstanding": none_value_extractor(bal_sheet.loc['Weighted average shares outstanding'].value),
            "revenue_breakdown": [{"class":revenue_bd.iloc[i,1],"category": revenue_bd.iloc[i,0], "amount": (revenue_bd.iloc[i,2])} for i in range (revenue_bd.shape[0])],
            "operating_expense_breakdown": [{"class":expense_bd.iloc[i,1],"category": expense_bd.iloc[i,0], "amount": int(-(expense_bd.iloc[i,2]))} for i in range (expense_bd.shape[0])]
            }

        balance_sheet = {
            "total_current_asset": none_value_extractor(bal_sheet.loc['Current Asset'].value),
            "total_non_current_asset": none_value_extractor(bal_sheet.loc['Non-Current Asset'].value),
            "total_asset": none_value_extractor(bal_sheet.loc['TOTAL ASSET'].value),
            "total_current_liabilities": none_value_extractor(bal_sheet.loc['Current Liabilities'].value),
            "total_non_current_liabilities": none_value_extractor(bal_sheet.loc['Non-Current Liabilities'].value),
            "total_liabilities": none_value_extractor(bal_sheet.loc['TOTAL LIABILITIES'].value),
            "total_equity": none_value_extractor(bal_sheet.loc["TOTAL SHAREHOLDER'S EQUITY"].value),
            "working_capital": none_value_extractor(bal_sheet.loc["Working Capital"].value)
            }

        try:
            capex = bal_sheet.loc['CAPITAL EXPENDITURE'].value
        except:
            try:
                capex = bal_sheet.loc['Net PP&E (current)'].value - bal_sheet.loc['Net PP&E (previous year)'].value + bal_sheet.loc['Depreciation expenses (current)'].value
            except:
                capex = None

        cash_flow = {
            "operating_cash_flow": none_value_extractor(cash_fl.loc['Cash Flows from Operating Activities'].value),
            "investing_cash_flow": none_value_extractor(cash_fl.loc['Cash Flows from Investing Activities'].value),
            "financing_cash_flow": none_value_extractor(cash_fl.loc['Cash Flows from Financing Activities'].value),
            "net_cash_flow": none_value_extractor(cash_fl.loc['NET INCREASE/DECREASED'].value),
            "capital_expenditure": none_value_extractor(capex),
            "free_cash_flow": none_value_extractor(cash_fl.loc['Cash Flows from Operating Activities'].value - capex if capex != None else None),
            }

        for metric in list(balance_sheet.keys()):
            balance_sheet[metric] = int(balance_sheet[metric])
            if currency == "USD":
                balance_sheet[metric] = int(balance_sheet[metric] * usd_idr_rate)

        for metric in list(cash_flow.keys()):
            if currency == "USD":
                try:
                    cash_flow[metric] = int(cash_flow[metric])
                    cash_flow[metric] = int(cash_flow[metric] * usd_idr_rate)
                except:
                    pass

        for metric in ['total_revenue', 'cost_of_revenue', 'gross_income', 'operating_expense', 'operating_income', "non_operating_income_or_loss", 'pretax_income', 'income_taxes', 'net_income', 'interest_expense_non_operating', 'ebit', 'ebitda']:
            input[metric] = int(input[metric])
            if currency == "USD":
                input[metric] = int(input[metric] * usd_idr_rate)

        for breakdown in ['revenue_breakdown', 'operating_expense_breakdown']:
            for item in input[breakdown]:
                item['amount'] = int(item['amount'])
                if currency == "USD":
                    item['amount'] = int(item['amount'] * usd_idr_rate)

        # Employee
        employee = df.iloc[:,19:21]
        employee.columns = ['key','value']

        employee = employee.dropna(subset="key").iloc[1:,:].set_index("key")

        employee.index = ["permanent_employee","contract_employee","others_employee","total_employee"]

        if employee.loc['total_employee'].value == 0:
            employee = None
        else:
            employee.value = employee.value.astype('Int64')
            employee = employee["value"].to_dict()

        # Industry parameter breakdown
        breakdown = df.iloc[37:52,6:8]

        breakdown.columns = ['key','value']

        # Define the expected groups
        expected_groups = ['Operating Statistics', 'Coal Resources', 'Coal Reserves']

        # Assign groups based on the key prefixes
        def assign_group(key):
            if pd.isna(key):
                return None
            for group in expected_groups:
                if key.startswith(group):
                    return group
            return None

        breakdown['group'] = breakdown['key'].apply(assign_group)
        breakdown['group'] = breakdown['group'].ffill()

        # Remove the group header rows
        breakdown = breakdown[~breakdown['key'].isin(expected_groups)]

        # Clean the keys by removing the group prefix
        def clean_key(row):
            group = row['group']
            key = row['key']
            if pd.isna(key):
                return key
            if key.startswith(group + '- '):
                return key[len(group + '- '):]
            elif key.startswith(group + ' - '):
                return key[len(group + ' - '):]
            else:
                return key

        breakdown['key'] = breakdown.apply(clean_key, axis=1)

        # Remove rows with NaN keys
        breakdown = breakdown.dropna(subset=['key'])

        if breakdown.shape[0] == 0:
            industry_breakdown = None
        else:
            breakdown = breakdown[~breakdown.value.isna()]
            industry_breakdown = breakdown.groupby('group').apply(
                lambda group: group.set_index('key')['value'].to_dict()
            ).to_dict()

        try:
            industry_breakdown["cost_of_revenue_breakdown"] = cost_of_revenue_breakdown
        except:
            pass

        # Coal Industry Breakdown
        try:
            # Get header positions from the header_rows dictionary
            headers = ['Sales Breakdown', 'Project Breakdown', 'Spec Breakdown']
            header_rows = {}

            data_section = df.iloc[52:, 6:]
            for idx, row in data_section.iterrows():
                first_val = row.iloc[0]
                if first_val in headers:
                    header_rows[first_val] = idx

            header_positions = list(header_rows.items())
            column_start = 6
            column_end = 19

            breakdowns = {}

            for i, (breakdown_name, header_row) in enumerate(header_positions):
                col_names_row = header_row + 1
                data_start_row = header_row + 2

                # Calculate data end row (next header row or end of dataframe)
                if i < len(header_positions) - 1:
                    data_end_row = header_positions[i + 1][1]  # Next header row
                else:
                    data_end_row = len(df)  # End of dataframe for last breakdown

                # Extract data
                bd = df.iloc[data_start_row:data_end_row, column_start:column_end].copy()
                bd.columns = df.iloc[col_names_row, column_start:column_end].values

                # Remove rows with all NaN and rows where first column is NaN
                bd = bd.dropna(how='all')
                bd = bd[bd.iloc[:, 0].notna()]

                breakdowns[breakdown_name] = bd

            # Create individual variables for easy access
            sales_bd = breakdowns['Sales Breakdown'][[col for col in breakdowns['Sales Breakdown'].columns if col is not None and not (isinstance(col, float) and np.isnan(col))]]
            sales_bd.columns = ["country","volume","volume_percentage","value_percentage_x","value","volume_percentage_x","value_percentage"]
            sales_bd.drop(columns=["volume_percentage_x","value_percentage_x"], inplace=True)
            sales_bd['volume_percentage'] = sales_bd['volume_percentage']/100
            sales_bd = sales_bd.replace({np.nan: None})

            project_bd = breakdowns['Project Breakdown'][[col for col in breakdowns['Project Breakdown'].columns if col is not None and not (isinstance(col, float) and np.isnan(col))]]
            project_bd.rename(columns={project_bd.columns[0]:'project'}, inplace=True)
            project_bd = project_bd.replace({np.nan: None})

            spec_bd = breakdowns['Spec Breakdown'][[col for col in breakdowns['Spec Breakdown'].columns if col is not None and not (isinstance(col, float) and np.isnan(col))]]
            spec_bd = spec_bd.replace({np.nan: None})

            industry_breakdown["project_breakdown"] = project_bd.set_index('project').to_dict(orient='index')

            industry_breakdown["sales_breakdown"] = sales_bd.set_index('country').to_dict(orient='index')

            industry_breakdown["spec_breakdown"] = spec_bd.set_index('product_name').to_dict(orient='index')

        except:
            pass

    return input,balance_sheet,cash_flow, employee, industry_breakdown

def generate_inputs(excel_file_path, usd_idr_rate, bank_company):
    inputs = []
    balance_sheets = []
    cash_flows = []
    employees = []
    industry_breakdowns = []
    sheet_names = _get_sheet_names(excel_file_path)
    for sheet_name in sheet_names:
        input, balance_sheet, cash_flow, employee, industry_breakdown = _process_sheet(excel_file_path, sheet_name, usd_idr_rate, bank_company)
        inputs.append(input)
        balance_sheets.append(balance_sheet)
        cash_flows.append(cash_flow)
        employees.append(employee)
        industry_breakdowns.append(industry_breakdown)

        print(f'Finish extracting manual input data of {sheet_name}')
    return inputs,balance_sheets,cash_flows, employees, industry_breakdowns

def validate_input(input, bank_company):
    error_messages = []

    def is_within_threshold(value, target, threshold):
        absolute_difference = abs(value - target)
        # avoid division by zero by adding a small number to the denominator
        return absolute_difference / abs(target + 1e-6) <= threshold
    if bank_company:
        if not is_within_threshold(
            (input["interest_income"] - input["interest_expense"]),
            input["net_interest_income"],
            0.02,
        ):
            error_messages.append(
                "Net Interest Income must be equal to Interest Income - Interest Expense"
            )

        if not is_within_threshold(
                (input["premium_income"] - input["premium_expense"]),
                input["net_premium_income"],
                0.02,
            ):
                error_messages.append(
                    "Net Premium Income must be equal to Premium Income - Premium Expense"
                )

        if not is_within_threshold(
                (input["net_interest_income"] + input["net_premium_income"] + input["non_interest_income"]),
                input["total_revenue"],
                0.02,
            ):
                error_messages.append(
                    "Total Revenue must be equal to Net Interest Income + Net Premium Income + Non Interest Income"
                )

        if not is_within_threshold(
                # total revenue - operating expense - provision = operating income
                (input["total_revenue"] - input["operating_expense"] - input["provision"]),
                input["operating_income"],
                0.02,
            ):
                error_messages.append(
                    "Operating Income must be equal to Total Revenue - Operating Expense - Provision for Impairment"
                )

        int_income_breakdown_total = sum(
            [ib["amount"] for ib in input["int_income_breakdown"]]
        )

        if not is_within_threshold(
            input["interest_income"], int_income_breakdown_total, 0.02
        ):
            error_messages.append(
                "Interest Income must be equal to the sum of interest income breakdown"
            )

    else:
        if not is_within_threshold(
            (input["total_revenue"] - input["cost_of_revenue"]),
            input["gross_income"],
            0.02,
        ):
            error_messages.append(
                "Total Revenue - Cost of Revenue must be equal to Gross Income"
            )

        if not is_within_threshold(
            (input["gross_income"] - input["operating_expense"]),
            input["operating_income"],
            0.02,
        ):
            error_messages.append(
                "Gross Income - Operating Expense must be equal to Operating Income"
            )

        # if not is_within_threshold(
        #     (input["pretax_income"] + input["interest_expense_non_operating"]),
        #     input["ebit"],
        #     0.02,
        # ):
        #     error_messages.append(
        #         "Pretax Income + Non Operating Interest Expense must be equal to EBIT"
        #     )
        revenue_breakdown_total = sum(
            [rb["amount"] for rb in input["revenue_breakdown"]]
        )
        if not is_within_threshold(input["total_revenue"], revenue_breakdown_total, 0.02):
            error_messages.append(
                "Total Revenue must be equal to the sum of revenue breakdown"
            )

    operating_expense_breakdown_total = sum(
        [oeb["amount"] for oeb in input["operating_expense_breakdown"]]
    )
    if not is_within_threshold(
        input["operating_expense"], operating_expense_breakdown_total, 0.02
    ):
        error_messages.append(
            "Operating Expense must be equal to the sum of operating expense breakdown"
        )

    return error_messages


def make_sankey_component(input, bank_company):
    def make_links():
        links = []

        if bank_company:
            for iib in input["int_income_breakdown"]:
                links.append(
                    {
                        "source": iib["category"],
                        "target": "Interest Income",
                        "value": iib["amount"],
                    }
                )

            links.append(
                {
                    "source": "Interest Income",
                    "target": "Net Interest Income",
                    "value": input["net_interest_income"],
                }
            )

            links.append(
                {
                    "source": "Interest Income",
                    "target": "Interest Expense",
                    "value": input["interest_expense"],
                }
            )

            links.append(
                {
                    "source": "Net Interest Income",
                    "target": "Total Revenue",
                    "value": input["net_interest_income"],
                }
            )

            if input["net_premium_income"] != 0:
                links.append(
                    {
                        "source": "Net Premium Income",
                        "target": "Total Revenue",
                        "value": input["net_premium_income"],
                    }
                )

            links.append(
                {
                    "source": "Non Interest Income",
                    "target": "Total Revenue",
                    "value": input["non_interest_income"],
                }
            )

            if input["operating_income"] >= 0:
                links.append(
                    {
                        "source": "Total Revenue",
                        "target": "Operating Income",
                        "value": input["operating_income"],
                    }
                )
                links.append(
                    {
                        "source": "Total Revenue",
                        "target": "Operating and Credit-Related Expenses",
                        "value": input["operating_expense"] + input["provision"],
                    }
                )
            else:
                links.append(
                    {
                        "source": "Operating Income",
                        "target": "Operating and Credit-Related Expenses",
                        "value": -(input["operating_income"]),
                    }
                )
                links.append(
                    {
                        "source": "Total Revenue",
                        "target": "Operating and Credit-Related Expenses",
                        "value": input["total_revenue"],
                    }
                )

            links.append(
                {
                    "source": "Operating and Credit-Related Expenses",
                    "target": "Operating Expense",
                    "value": input["operating_expense"],
                }
            )

            links.append(
                {
                    "source": "Operating and Credit-Related Expenses",
                    "target": "Provision for Impairment",
                    "value": input["provision"],
                }
            )

            for oeb in input["operating_expense_breakdown"]:
                links.append(
                    {
                        "source": "Operating Expense",
                        "target": oeb["category"],
                        "value": oeb["amount"],
                    }
                )

        else:
            for rb in input["revenue_breakdown"]:
                links.append(
                    {
                        "source": rb["category"],
                        "target": "Total Revenue",
                        "value": rb["amount"],
                    }
                )

            if input["gross_income"] >= 0:
                links.append(
                    {
                        "source": "Total Revenue",
                        "target": "Cost of Revenue",
                        "value": input["cost_of_revenue"],
                    }
                )

                links.append(
                    {
                        "source": "Total Revenue",
                        "target": "Gross Profit",
                        "value": input["gross_income"],
                    }
                )
                if input["operating_income"] >= 0:
                    links.append(
                        {
                            "source": "Gross Profit",
                            "target": "Operating Income",
                            "value": input["operating_income"],
                        }
                    )
                    links.append(
                        {
                            "source": "Gross Profit",
                            "target": "Operating Expense",
                            "value": input["operating_expense"],
                        }
                    )
                else:
                    links.append(
                        {
                            "source": "Operating Income",
                            "target": "Operating Expense",
                            "value": -(input["operating_income"]),
                        }
                    )
                    links.append(
                        {
                            "source": "Gross Profit",
                            "target": "Operating Expense",
                            "value": input["gross_income"],
                        }
                    )

            else:
                links.append(
                    {
                        "source": "Total Revenue",
                        "target": "Cost of Revenue",
                        "value": input["total_revenue"],
                    }
                )
                links.append(
                    {
                        "source": "Gross Profit",
                        "target": "Cost of Revenue",
                        "value": -(input["gross_income"]),
                    }
                )
                links.append(
                    {
                        "source": "Operating Income",
                        "target": "Gross Profit",
                        "value": -(input["gross_income"]),
                    }
                )
                links.append(
                    {
                        "source": "Operating Income",
                        "target": "Operating Expense",
                        "value": input["operating_expense"],
                    }
                )

            for oeb in input["operating_expense_breakdown"]:
                links.append(
                    {
                        "source": "Operating Expense",
                        "target": oeb["category"],
                        "value": oeb["amount"],
                    }
                )

        return links

    def make_nodes():
        red = "hsl(0, 100%, 50%)"
        orange = "hsl(39, 100%, 50%)"
        light_blue = "hsl(195, 53%, 79%)"
        dark_blue = "hsl(240, 100%, 50%)"

        nodes = []

        if bank_company:
            for iib in input["int_income_breakdown"]:
                nodes.append({"id": iib["category"], "nodeColor": light_blue})
            nodes.append({"id": "Interest Income", "nodeColor": light_blue})
            nodes.append({"id": "Interest Expense", "nodeColor": orange})
            nodes.append({"id": "Net Interest Income", "nodeColor": light_blue})
            if input["net_premium_income"] != 0:
                nodes.append({"id": "Net Premium Income", "nodeColor": light_blue})
            nodes.append({"id": "Non Interest Income", "nodeColor": light_blue})
            nodes.append({"id": "Total Revenue", "nodeColor": light_blue})

            if input["operating_income"] >= 0:
                nodes.append({"id": "Operating Income", "nodeColor": dark_blue})
            else:
                nodes.append({"id": "Operating Income", "nodeColor": red})

            nodes.append({"id": "Operating and Credit-Related Expenses", "nodeColor": orange})
            nodes.append({"id": "Operating Expense", "nodeColor": orange})
            nodes.append({"id": "Provision for Impairment", "nodeColor": orange})

            for oeb in input["operating_expense_breakdown"]:
                nodes.append({"id": oeb["category"], "nodeColor": orange})

        else:
            for rb in input["revenue_breakdown"]:
                nodes.append({"id": rb["category"], "nodeColor": light_blue})
            nodes.append({"id": "Total Revenue", "nodeColor": light_blue})

            nodes.append({"id": "Cost of Revenue", "nodeColor": orange})
            if input["gross_income"] >= 0:
                nodes.append({"id": "Gross Profit", "nodeColor": dark_blue})
            else:
                nodes.append({"id": "Gross Profit", "nodeColor": red})

            if input["operating_income"] >= 0:
                nodes.append({"id": "Operating Income", "nodeColor": dark_blue})
            else:
                nodes.append({"id": "Operating Income", "nodeColor": red})
            nodes.append({"id": "Operating Expense", "nodeColor": orange})

            for oeb in input["operating_expense_breakdown"]:
                nodes.append({"id": oeb["category"], "nodeColor": orange})

        return nodes

    output = {"nodes": make_nodes(), "links": make_links()}
    return output

def process_excel_file(file_path, financial_year, usd_idr_rate, bank_company):
    inputs,balance_sheets,cash_flows,employees, industry_breakdowns = generate_inputs(file_path, usd_idr_rate, bank_company)
    records = []

    for input in range(0,len(inputs)):
        errors = validate_input(inputs[input], bank_company)
        input_copy = inputs[input].copy()

        symbol = input_copy.pop('symbol')
        source_url = input_copy.pop('url')

        if len(errors) > 0:
            print(f"Errors for {symbol}: {errors}")
            output = None
        else:
            output = make_sankey_component(inputs[input], bank_company)

        source_url = source_url
        record = {
            'symbol': symbol + ".JK",
            'financial_year': financial_year,
            'sankey_component': output,
            'source_url': source_url,
            'income_stmt_metrics': input_copy,
            'balance_sheet_metrics': balance_sheets[input],
            'cash_flow_metrics': cash_flows[input],
            'employee_breakdown': employees[input],
            'industry_breakdown': industry_breakdowns[input],
            'updated_on': pd.to_datetime('now').strftime("%Y-%m-%d %H:%M:%S")
        }
        records.append(record)

    return inputs, records

## 2024 FY version

In [ ]:
def _get_sheet_names(excel_file_path):
    excel_file = pd.ExcelFile(excel_file_path)
    sheet_names = excel_file.sheet_names
    # sheet_names = ['BMRI']
    return sheet_names


def _filter_others_keys(row: pd.Series):
    """
    Removes keys containing \"Other\" by replacing their value with NaN,
    and return its values as series to be summed

    MODIFIES the DataFrame directly!
    """
    if re.search("^[Oo]ther", row.name):
        value = row.value
        row.value = np.NaN
        return value

def none_value_extractor(metrics):
    if pd.isna(metrics):
        return None
    else:
        return metrics

def _reduce_others_keys(df: pd.DataFrame):
    """
    Groups all \"Other\" breakdown entries into a single \"Others\" by summing them up

    MODIFIES the DataFrame directly!
    """

    others_rev = df[df.index.str.contains("^[Oo]ther")]
    df = df[~df.index.str.contains("^[Oo]ther")]

    if others_rev.shape[0] > 0:
        df.loc["Others", "value"] = int(others_rev["value"].sum())
        df.loc["Others", "category"] = others_rev['category'].iloc[0]
        return df
    else:
        return df

def _process_sheet(excel_file_path, sheet_name, usd_idr_rate, bank_company):
    df = pd.read_excel(excel_file_path, sheet_name=sheet_name, header=None)

    data = df[[0,1]]
    data = data.rename(columns={0:'key', 1:'value'})
    data = data.dropna(subset=['key'])
    data = data.set_index('key')
    metadata = data.loc['symbol':'currency'].copy()
    # replace NaN value with None for postgrest json
    metadata = metadata.replace({np.nan: None})

    currency = metadata.loc['currency'].value

    if bank_company:

        # Balance sheet
        bal_sheet = df.iloc[:,13:15]
        bal_sheet.columns = ["key",'value']
        bal_sheet['group'] = bal_sheet['key'].where(bal_sheet['value'].isna()).ffill()
        bal_sheet = bal_sheet.set_index("key")

        # Non-loan-asset breakdown
        non_loan_asset_bd = df[[15,16,17]]
        non_loan_asset_bd = non_loan_asset_bd.rename(columns={15:'class',16:'category', 17:'amount'}).dropna(subset=['category'])
        non_loan_asset_bd = non_loan_asset_bd.set_index('category')
        non_loan_asset_bd = non_loan_asset_bd.drop(index=["sum", "match",'Breakdown of: Non-loan asset']).reset_index()

        balance_sheet = {
            "gross_loan": none_value_extractor(bal_sheet.loc['Gross Loan'].value),
            "allowance_for_loans": none_value_extractor(bal_sheet.loc['Allowance for Loans'].value),
            "net_loan": none_value_extractor(bal_sheet.loc['NET LOAN'].value),
            'earning_asset': none_value_extractor(non_loan_asset_bd[non_loan_asset_bd['class'] == "Earning Asset"].amount.sum() + bal_sheet.loc['NET LOAN'].value if pd.isna(bal_sheet.loc['Total Earning Asset'].value) else bal_sheet.loc['Total Earning Asset'].value),
            "non_loan_asset": none_value_extractor(bal_sheet.loc['Non-Loan Asset'].value),
            "total_asset": none_value_extractor(bal_sheet.loc['TOTAL ASSET'].value),
            "current_account": none_value_extractor(bal_sheet.loc['Current Account'].value),
            "savings_account": none_value_extractor(bal_sheet.loc['Savings Account'].value),
            "time_deposit": none_value_extractor(bal_sheet.loc['Time Deposits'].value),
            "total_deposit": none_value_extractor(bal_sheet.loc['TOTAL DEPOSIT'].value),
            "other_interest_bearing_liabilities": none_value_extractor(bal_sheet.loc['Other Interest-Bearing Liabilities'].value),
            "non_interest_bearing_liabilities": none_value_extractor(bal_sheet.loc['Non-Interest-Bearning Liabilities'].value),
            "total_liabilities": none_value_extractor(bal_sheet.loc['TOTAL LIABILITIES'].value),
            "total_equity": none_value_extractor(bal_sheet.loc["TOTAL SHAREHOLDER'S EQUITY"].value),
            "core_capital_tier1": none_value_extractor(bal_sheet.loc['Core Capital (Tier 1)'].value),
            "supplementary_capital_tier2": none_value_extractor(bal_sheet.loc['Supplementary Capital (Tier 2)'].value),
            "total_capital": none_value_extractor(bal_sheet.loc['TOTAL CAPITAL'].value),
            "credit_rwa": none_value_extractor(bal_sheet.loc['RWAs Considering Credit Risk'].value),
            "market_rwa": none_value_extractor(bal_sheet.loc['RWAs Considering Market Risk'].value),
            "operational_rwa": none_value_extractor(bal_sheet.loc['RWAs Considering Operational Risk'].value),
            "total_risk_weighted_asset": none_value_extractor(bal_sheet.loc['TOTAL RISK-WEIGHTED ASSETS'].value)
            }

        for metric in list(balance_sheet.keys()):
            try:
                balance_sheet[metric] = int(balance_sheet[metric])
                if currency == "USD":
                    balance_sheet[metric] = int(balance_sheet[metric] * usd_idr_rate)
            except:
                pass

        # Income Statement
        income_stmt = data.loc['interest income':'net income'].copy()
        income_stmt['value'] = income_stmt['value'].fillna(0)

        int_income_bd = df[[2,3,4]]
        int_income_bd = int_income_bd.rename(columns={2:'category',3:'key', 4:'value'}).dropna(subset=['key'])
        int_income_bd = int_income_bd.set_index('key')
        int_income_bd = int_income_bd.drop(index=["sum", "match",'subsector','Breakdown of: total revenue','Year'])
        int_income_bd = int_income_bd.dropna(subset=['value'])
        # Sum up all "Other" expenses into a single entry of "Others"
        int_income_bd = _reduce_others_keys(int_income_bd)
        int_income_bd = int_income_bd.reset_index()

        expense_bd = df[[5,6,7]]
        expense_bd = expense_bd.rename(columns={5:'category',6:'key', 7:'value'}).dropna(subset=['key'])
        expense_bd = expense_bd.set_index('key')
        expense_bd = expense_bd.drop(index=["sum", "match",'Breakdown of: operating expenses'])
        # check for positive expense; expense is standardized as having negative values only
        if (expense_bd['value'].values > 0).any():
            raise ValueError(f'Positive value detected in expense breakdown of sheet {sheet_name}')
        # Sum up all "Other" expenses into a single entry of "Others"
        expense_bd = _reduce_others_keys(expense_bd)
        # rename the expense index if any match is found in income breakdown
        for expense_idx in expense_bd.index:
            if expense_idx in int_income_bd['key'].values:
                expense_bd.rename(index={expense_idx: expense_idx + ' (expense)'}, inplace=True)
            elif expense_idx == "Others":
                expense_bd.rename(index={expense_idx: expense_idx + ' (expense)'}, inplace=True)

        expense_bd = expense_bd.reset_index()

        try:
            non_operating_income = income_stmt.loc['net non operating income/(expenses)'].value
        except:
            non_operating_income = income_stmt.loc['net non operating income'].value

        input = {
            "symbol": metadata.loc['symbol'].value,
            "url": metadata.loc['url'].value,
            "interest_income": none_value_extractor(income_stmt.loc['interest income'].value),
            "interest_expense": none_value_extractor(-(income_stmt.loc['interest expenses'].value)),
            "net_interest_income": none_value_extractor(income_stmt.loc['net interest income'].value),
            "premium_income": none_value_extractor(income_stmt.loc['premium income'].value),
            "premium_expense": none_value_extractor(-(income_stmt.loc['premium expense'].value)),
            "net_premium_income": none_value_extractor(income_stmt.loc['net premium income'].value),
            "non_interest_income": none_value_extractor(income_stmt.loc['non interest income'].value),
            "total_revenue": none_value_extractor(income_stmt.loc['total revenue'].value),
            "operating_expense": none_value_extractor(-(income_stmt.loc['operating expenses'].value)),
            "provision": none_value_extractor(-(income_stmt.loc['provision for impairment'].value)),
            "operating_income": none_value_extractor(income_stmt.loc['net operating income'].value),
            "non_operating_income_or_loss": none_value_extractor(non_operating_income),
            "pretax_income": none_value_extractor(income_stmt.loc['pretax income'].value),
            "income_taxes": none_value_extractor(-(income_stmt.loc['tax'].value)),
            "net_income": none_value_extractor(income_stmt.loc['net income'].value),
            "diluted_shares_outstanding": none_value_extractor(bal_sheet.loc['Weighted average shares outstanding'].value if bal_sheet.loc['Weighted average shares outstanding'].value != 0 else None),
            "int_income_breakdown": [{"class":int_income_bd.iloc[i,1],"category": int_income_bd.iloc[i,0], "amount": (int_income_bd.iloc[i,2])} for i in range (int_income_bd.shape[0])],
            "operating_expense_breakdown": [{"class":expense_bd.iloc[i,1],"category": expense_bd.iloc[i,0], "amount": int(-(expense_bd.iloc[i,2]))} for i in range (expense_bd.shape[0])]
            }

        for metric in ['interest_income', 'interest_expense', 'net_interest_income', 'premium_income', 'premium_expense', 'net_premium_income', 'non_interest_income', 'total_revenue', 'operating_expense', 'provision', 'operating_income',"non_operating_income_or_loss", 'pretax_income', 'income_taxes', 'net_income']:
            input[metric] = int(input[metric])
            if currency == "USD":
                input[metric] = int(input[metric] * usd_idr_rate)

        for breakdown in ['int_income_breakdown', 'operating_expense_breakdown']:
            for item in input[breakdown]:
                item['amount'] = int(item['amount'])
                if currency == "USD":
                    item['amount'] = int(item['amount'] * usd_idr_rate)

        # Industry Breakdown
        # try:
        #     sector_loan_breakdown = bal_sheet[bal_sheet.group == "Breakdown of: gross loan (by economic sector)"]
        #     sector_loan_breakdown = sector_loan_breakdown.drop(index=["sum", 'Breakdown of: gross loan (by economic sector)'])
        #     sector_loan_breakdown = sector_loan_breakdown.dropna(subset=['value'])
        #     sector_loan_breakdown['group'] = "gross_loan"
        # except:
        sector_loan_breakdown = bal_sheet[bal_sheet.group == "Breakdown of: loan receivable (by economic sector)"]
        sector_loan_breakdown = sector_loan_breakdown.drop(index=["sum", 'Breakdown of: loan receivable (by economic sector)'])
        sector_loan_breakdown = sector_loan_breakdown.dropna(subset=['value'])
        sector_loan_breakdown['group'] = "loan_receivable"

        try:
            special_mention_loan = bal_sheet.loc['Special Mention Loan (Loan receivable)'].value
        except:
            special_mention_loan = bal_sheet.loc['Special Mention Loan'].value

        try:
            restructured_loan = bal_sheet.loc['Restructured Loan (current)'].value
        except:
            restructured_loan = bal_sheet.loc['Restructured Loan'].value

        industry_breakdown = {
            "loan_at_risk":{
            "Special Mention Loan": special_mention_loan,
            "Restructured Loan (current)": restructured_loan,
            "Non-performing Loan (NPL)": bal_sheet.loc['Non-performing Loan (NPL)'].value
            },
            "loan_by_economic_sectors": sector_loan_breakdown.reset_index().to_dict(orient='records'),
            "non_loan_asset": non_loan_asset_bd.to_dict(orient='records')
        }

        # Cash Flow
        cash_fl = df[[19,20]]#.dropna()
        cash_fl.columns = ["key",'value']
        cash_fl = cash_fl.set_index("key")

        cash_flow = {
                    "high_quality_liquid_asset": none_value_extractor(cash_fl.loc['Total High Quality Liquid Asset (HQLA)'].value),
                    "cash_outflow": none_value_extractor(cash_fl.loc['Cash Outflow'].value),
                    "cash_inflow": none_value_extractor(cash_fl.loc['Cash Inflow'].value),
                    "end_cash_position": none_value_extractor(cash_fl.loc['TOTAL NET CASH OUTFLOWS'].value),
                    "operating_cash_flow": cash_fl.loc['Cash Flows from Operating Activities'].value,
                    "investing_cash_flow": cash_fl.loc['Cash Flows from Investing Activities'].value,
                    "financing_cash_flow": cash_fl.loc['Cash Flows from Financing Activities'].value,
                    "net_cash_flow": cash_fl.loc['NET INCREASE/DECREASED'].value,
                    "realized_capital_goods_investment": none_value_extractor(cash_fl.loc['REALIZED CAPITAL GOODS INVESTMENT'].value),
                    "free_cash_flow": none_value_extractor(cash_fl.loc['Cash Flows from Operating Activities'].value - cash_fl.loc['REALIZED CAPITAL GOODS INVESTMENT'].value),
                    }

        for metric in list(cash_flow.keys()):
            if currency == "USD":
                try:
                    cash_flow[metric] = int(cash_flow[metric])
                    cash_flow[metric] = int(cash_flow[metric] * usd_idr_rate)
                except:
                    pass

        # Employee
        employee = df.iloc[:,22:24]
        employee.columns = ['key','value']

        employee = employee.dropna(subset="key").iloc[1:,:].set_index("key")

        employee.index = ["permanent_employee","contract_employee","others_employee","total_employee"]

        if employee.loc['total_employee'].value == 0:
            employee = None
        else:
            employee.value = employee.value.astype('Int64')
            employee = employee["value"].to_dict()

    else:  # NOT bank company
        # Income Statement Metrics
        income_stmt = data.loc['total revenue':'ebitda'].copy()
        income_stmt['value'] = income_stmt['value'].fillna(0)

        revenue_bd = df[[2,3,4]]
        revenue_bd = revenue_bd.rename(columns={2:'category',3:'key', 4:'value'}).dropna(subset=['key'])
        revenue_bd = revenue_bd.set_index('key')
        revenue_bd = revenue_bd.drop(index=["sum", "match",'subsector','Breakdown of: total revenue','Year'])

        ## Sum up all "Other" expenses into a single entry of "Others"
        revenue_bd =_reduce_others_keys(revenue_bd)
        revenue_bd.reset_index(inplace=True)

        # Balance sheet Metrics
        bal_sheet = df.iloc[:50,13:15]
        bal_sheet.columns = ["key",'value']

        bal_sheet = bal_sheet.set_index("key")

        # Cash Flow Metrics
        cash_fl = df.iloc[:,16:18]
        cash_fl.columns = ["key",'value']

        cash_fl = cash_fl.dropna(subset=['key'])

        cash_fl = cash_fl.set_index("key")


        try:
            expense_bd = df[[5,6,7,8]]
            expense_bd = expense_bd.rename(columns={5:'category',6:'key', 7:'value', 8:'group'})

            expense_bd = expense_bd[~expense_bd.key.isin(["sum","match",'Breakdown of: operating expenses'])]

            expense_bd["group"] = expense_bd["group"].ffill()

            expense_bd = expense_bd.dropna()

            expense_bd["key"] = expense_bd["group"] + '-' + expense_bd["key"]

            expense_bd = expense_bd.drop(['group'], axis=1)
        except:
            expense_bd = df[[5,6,7]]
            expense_bd = expense_bd.rename(columns={5:'category',6:'key', 7:'value'}).dropna(subset=['key'])
            expense_bd = expense_bd.set_index('key')
            expense_bd = expense_bd.drop(index=["sum", "match",'Breakdown of: operating expenses'])
            # check for positive expense; expense is standardized as having negative values only
            if (expense_bd['value'].values > 0).any():
                raise ValueError(f'Positive value detected in expense breakdown of sheet {sheet_name}')
            # Sum up all "Other" expenses into a single entry of "Others"
            expense_bd = _reduce_others_keys(expense_bd)
            # rename the expense index if any match is found in income breakdown
            for expense_idx in expense_bd.index:
                if expense_idx in revenue_bd['key'].values:
                    expense_bd.rename(index={expense_idx: expense_idx + ' (expense)'}, inplace=True)
                elif expense_idx == "Others":
                    expense_bd.rename(index={expense_idx: expense_idx + ' (expense)'}, inplace=True)

            expense_bd.reset_index(inplace=True)

            expense_bd["value"] = expense_bd["value"].astype('int')

        input = {
            "symbol": metadata.loc['symbol'].value,
            "url": metadata.loc['url'].value,
            "total_revenue": none_value_extractor(income_stmt.loc['total revenue'].value),
            "cost_of_revenue": none_value_extractor(-(income_stmt.loc['cost of revenue'].value)),
            "gross_income": none_value_extractor(income_stmt.loc['gross income'].value),
            "operating_expense": none_value_extractor(-(income_stmt.loc['operating expenses'].value)),
            "operating_income": none_value_extractor(income_stmt.loc['net operating income'].value),
            "non_operating_income_or_loss": none_value_extractor(income_stmt.loc['net non operating income/(expenses)'].value),
            "pretax_income": none_value_extractor(income_stmt.loc['pretax income'].value),
            "income_taxes":  none_value_extractor(-(income_stmt.loc['tax'].value)),
            "net_income": none_value_extractor(income_stmt.loc['net income'].value),
            "interest_expense_non_operating": none_value_extractor(-(income_stmt.loc['non operating interest expense'].value)),
            "ebit": none_value_extractor(income_stmt.loc['ebit'].value),
            "ebitda": none_value_extractor(income_stmt.loc['ebit'].value + income_stmt.loc['depreciation and amortization'].value if pd.isna(income_stmt.loc['ebitda'].value) else income_stmt.loc['ebitda'].value),
            "diluted_shares_outstanding": none_value_extractor(bal_sheet.loc['Weighted average shares outstanding'].value),
            "revenue_breakdown": [{"class":revenue_bd.iloc[i,1],"category": revenue_bd.iloc[i,0], "amount": (revenue_bd.iloc[i,2])} for i in range (revenue_bd.shape[0])],
            "operating_expense_breakdown": [{"class":expense_bd.iloc[i,1],"category": expense_bd.iloc[i,0], "amount": int(-(expense_bd.iloc[i,2]))} for i in range (expense_bd.shape[0])]
            }

        balance_sheet = {
            "total_current_asset": none_value_extractor(bal_sheet.loc['Current Asset'].value),
            "total_non_current_asset": none_value_extractor(bal_sheet.loc['Non-Current Asset'].value),
            "total_asset": none_value_extractor(bal_sheet.loc['TOTAL ASSET'].value),
            "total_current_liabilities": none_value_extractor(bal_sheet.loc['Current Liabilities'].value),
            "total_non_current_liabilities": none_value_extractor(bal_sheet.loc['Non-Current Liabilities'].value),
            "total_liabilities": none_value_extractor(bal_sheet.loc['TOTAL LIABILITIES'].value),
            "total_equity": none_value_extractor(bal_sheet.loc["TOTAL SHAREHOLDER'S EQUITY"].value),
            "working_capital": none_value_extractor(bal_sheet.loc["Working Capital"].value)
            }

        try:
            capex = bal_sheet.loc['CAPITAL EXPENDITURE'].value
        except:
            try:
                capex = bal_sheet.loc['Net PP&E (current)'].value - bal_sheet.loc['Net PP&E (previous year)'].value + bal_sheet.loc['Depreciation expenses (current)'].value
            except:
                capex = None

        cash_flow = {
            "operating_cash_flow": none_value_extractor(cash_fl.loc['Cash Flows from Operating Activities'].value),
            "investing_cash_flow": none_value_extractor(cash_fl.loc['Cash Flows from Investing Activities'].value),
            "financing_cash_flow": none_value_extractor(cash_fl.loc['Cash Flows from Financing Activities'].value),
            "net_cash_flow": none_value_extractor(cash_fl.loc['NET INCREASE/DECREASED'].value),
            "capital_expenditure": none_value_extractor(capex),
            "free_cash_flow": none_value_extractor(cash_fl.loc['Cash Flows from Operating Activities'].value - capex if capex != None else None),
            }

        for metric in list(balance_sheet.keys()):
            balance_sheet[metric] = int(balance_sheet[metric])
            if currency == "USD":
                balance_sheet[metric] = int(balance_sheet[metric] * usd_idr_rate)

        for metric in list(cash_flow.keys()):
            if currency == "USD":
                try:
                    cash_flow[metric] = int(cash_flow[metric])
                    cash_flow[metric] = int(cash_flow[metric] * usd_idr_rate)
                except:
                    pass

        for metric in ['total_revenue', 'cost_of_revenue', 'gross_income', 'operating_expense', 'operating_income', "non_operating_income_or_loss", 'pretax_income', 'income_taxes', 'net_income', 'interest_expense_non_operating', 'ebit', 'ebitda']:
            input[metric] = int(input[metric])
            if currency == "USD":
                input[metric] = int(input[metric] * usd_idr_rate)

        for breakdown in ['revenue_breakdown', 'operating_expense_breakdown']:
            for item in input[breakdown]:
                item['amount'] = int(item['amount'])
                if currency == "USD":
                    item['amount'] = int(item['amount'] * usd_idr_rate)

        # Employee
        employee = df.iloc[:,19:21]
        employee.columns = ['key','value']

        employee = employee.dropna(subset="key").iloc[1:,:].set_index("key")

        employee.index = ["permanent_employee","contract_employee","others_employee","total_employee"]

        if employee.loc['total_employee'].value == 0:
            employee = None
        else:
            employee.value = employee.value.astype('Int64')
            employee = employee["value"].to_dict()

        # Industry parameter breakdown
        breakdown = df.iloc[51:,13:15]

        breakdown.columns = ['key','value']
        breakdown['group'] = breakdown['key'].where(breakdown['value'].isna()).ffill()
        breakdown.dropna(subset="value", inplace=True)

        breakdown = breakdown[breakdown.value != 0]

        if breakdown.shape[0] == 0:
            industry_breakdown = None
        else:
            industry_breakdown = breakdown.groupby('group').apply(
                lambda group: group.set_index('key')['value'].to_dict()
            ).to_dict()

    return input,balance_sheet,cash_flow, employee, industry_breakdown

def generate_inputs(excel_file_path, usd_idr_rate, bank_company):
    inputs = []
    balance_sheets = []
    cash_flows = []
    employees = []
    industry_breakdowns = []
    sheet_names = _get_sheet_names(excel_file_path)
    for sheet_name in sheet_names:
        input, balance_sheet, cash_flow, employee, industry_breakdown = _process_sheet(excel_file_path, sheet_name, usd_idr_rate, bank_company)
        inputs.append(input)
        balance_sheets.append(balance_sheet)
        cash_flows.append(cash_flow)
        employees.append(employee)
        industry_breakdowns.append(industry_breakdown)

        print(f'Finish extracting manual input data of {sheet_name}')
    return inputs,balance_sheets,cash_flows, employees, industry_breakdowns

def validate_input(input, bank_company):
    error_messages = []

    def is_within_threshold(value, target, threshold):
        absolute_difference = abs(value - target)
        # avoid division by zero by adding a small number to the denominator
        return absolute_difference / abs(target + 1e-6) <= threshold
    if bank_company:
        if not is_within_threshold(
            (input["interest_income"] - input["interest_expense"]),
            input["net_interest_income"],
            0.02,
        ):
            error_messages.append(
                "Net Interest Income must be equal to Interest Income - Interest Expense"
            )

        if not is_within_threshold(
                (input["premium_income"] - input["premium_expense"]),
                input["net_premium_income"],
                0.02,
            ):
                error_messages.append(
                    "Net Premium Income must be equal to Premium Income - Premium Expense"
                )

        if not is_within_threshold(
                (input["net_interest_income"] + input["net_premium_income"] + input["non_interest_income"]),
                input["total_revenue"],
                0.02,
            ):
                error_messages.append(
                    "Total Revenue must be equal to Net Interest Income + Net Premium Income + Non Interest Income"
                )

        if not is_within_threshold(
                # total revenue - operating expense - provision = operating income
                (input["total_revenue"] - input["operating_expense"] - input["provision"]),
                input["operating_income"],
                0.02,
            ):
                error_messages.append(
                    "Operating Income must be equal to Total Revenue - Operating Expense - Provision for Impairment"
                )

        int_income_breakdown_total = sum(
            [ib["amount"] for ib in input["int_income_breakdown"]]
        )

        if not is_within_threshold(
            input["interest_income"], int_income_breakdown_total, 0.02
        ):
            error_messages.append(
                "Interest Income must be equal to the sum of interest income breakdown"
            )

    else:
        if not is_within_threshold(
            (input["total_revenue"] - input["cost_of_revenue"]),
            input["gross_income"],
            0.02,
        ):
            error_messages.append(
                "Total Revenue - Cost of Revenue must be equal to Gross Income"
            )

        if not is_within_threshold(
            (input["gross_income"] - input["operating_expense"]),
            input["operating_income"],
            0.02,
        ):
            error_messages.append(
                "Gross Income - Operating Expense must be equal to Operating Income"
            )

        # if not is_within_threshold(
        #     (input["pretax_income"] + input["interest_expense_non_operating"]),
        #     input["ebit"],
        #     0.02,
        # ):
        #     error_messages.append(
        #         "Pretax Income + Non Operating Interest Expense must be equal to EBIT"
        #     )
        revenue_breakdown_total = sum(
            [rb["amount"] for rb in input["revenue_breakdown"]]
        )
        if not is_within_threshold(input["total_revenue"], revenue_breakdown_total, 0.02):
            error_messages.append(
                "Total Revenue must be equal to the sum of revenue breakdown"
            )

    operating_expense_breakdown_total = sum(
        [oeb["amount"] for oeb in input["operating_expense_breakdown"]]
    )
    if not is_within_threshold(
        input["operating_expense"], operating_expense_breakdown_total, 0.02
    ):
        error_messages.append(
            "Operating Expense must be equal to the sum of operating expense breakdown"
        )

    return error_messages


def make_sankey_component(input, bank_company):
    def make_links():
        links = []

        if bank_company:
            for iib in input["int_income_breakdown"]:
                links.append(
                    {
                        "source": iib["category"],
                        "target": "Interest Income",
                        "value": iib["amount"],
                    }
                )

            links.append(
                {
                    "source": "Interest Income",
                    "target": "Net Interest Income",
                    "value": input["net_interest_income"],
                }
            )

            links.append(
                {
                    "source": "Interest Income",
                    "target": "Interest Expense",
                    "value": input["interest_expense"],
                }
            )

            links.append(
                {
                    "source": "Net Interest Income",
                    "target": "Total Revenue",
                    "value": input["net_interest_income"],
                }
            )

            if input["net_premium_income"] != 0:
                links.append(
                    {
                        "source": "Net Premium Income",
                        "target": "Total Revenue",
                        "value": input["net_premium_income"],
                    }
                )

            links.append(
                {
                    "source": "Non Interest Income",
                    "target": "Total Revenue",
                    "value": input["non_interest_income"],
                }
            )

            if input["operating_income"] >= 0:
                links.append(
                    {
                        "source": "Total Revenue",
                        "target": "Operating Income",
                        "value": input["operating_income"],
                    }
                )
                links.append(
                    {
                        "source": "Total Revenue",
                        "target": "Operating and Credit-Related Expenses",
                        "value": input["operating_expense"] + input["provision"],
                    }
                )
            else:
                links.append(
                    {
                        "source": "Operating Income",
                        "target": "Operating and Credit-Related Expenses",
                        "value": -(input["operating_income"]),
                    }
                )
                links.append(
                    {
                        "source": "Total Revenue",
                        "target": "Operating and Credit-Related Expenses",
                        "value": input["total_revenue"],
                    }
                )

            links.append(
                {
                    "source": "Operating and Credit-Related Expenses",
                    "target": "Operating Expense",
                    "value": input["operating_expense"],
                }
            )

            links.append(
                {
                    "source": "Operating and Credit-Related Expenses",
                    "target": "Provision for Impairment",
                    "value": input["provision"],
                }
            )

            for oeb in input["operating_expense_breakdown"]:
                links.append(
                    {
                        "source": "Operating Expense",
                        "target": oeb["category"],
                        "value": oeb["amount"],
                    }
                )

        else:
            for rb in input["revenue_breakdown"]:
                links.append(
                    {
                        "source": rb["category"],
                        "target": "Total Revenue",
                        "value": rb["amount"],
                    }
                )

            if input["gross_income"] >= 0:
                links.append(
                    {
                        "source": "Total Revenue",
                        "target": "Cost of Revenue",
                        "value": input["cost_of_revenue"],
                    }
                )

                links.append(
                    {
                        "source": "Total Revenue",
                        "target": "Gross Profit",
                        "value": input["gross_income"],
                    }
                )
                if input["operating_income"] >= 0:
                    links.append(
                        {
                            "source": "Gross Profit",
                            "target": "Operating Income",
                            "value": input["operating_income"],
                        }
                    )
                    links.append(
                        {
                            "source": "Gross Profit",
                            "target": "Operating Expense",
                            "value": input["operating_expense"],
                        }
                    )
                else:
                    links.append(
                        {
                            "source": "Operating Income",
                            "target": "Operating Expense",
                            "value": -(input["operating_income"]),
                        }
                    )
                    links.append(
                        {
                            "source": "Gross Profit",
                            "target": "Operating Expense",
                            "value": input["gross_income"],
                        }
                    )

            else:
                links.append(
                    {
                        "source": "Total Revenue",
                        "target": "Cost of Revenue",
                        "value": input["total_revenue"],
                    }
                )
                links.append(
                    {
                        "source": "Gross Profit",
                        "target": "Cost of Revenue",
                        "value": -(input["gross_income"]),
                    }
                )
                links.append(
                    {
                        "source": "Operating Income",
                        "target": "Gross Profit",
                        "value": -(input["gross_income"]),
                    }
                )
                links.append(
                    {
                        "source": "Operating Income",
                        "target": "Operating Expense",
                        "value": input["operating_expense"],
                    }
                )

            for oeb in input["operating_expense_breakdown"]:
                links.append(
                    {
                        "source": "Operating Expense",
                        "target": oeb["category"],
                        "value": oeb["amount"],
                    }
                )

        return links

    def make_nodes():
        red = "hsl(0, 100%, 50%)"
        orange = "hsl(39, 100%, 50%)"
        light_blue = "hsl(195, 53%, 79%)"
        dark_blue = "hsl(240, 100%, 50%)"

        nodes = []

        if bank_company:
            for iib in input["int_income_breakdown"]:
                nodes.append({"id": iib["category"], "nodeColor": light_blue})
            nodes.append({"id": "Interest Income", "nodeColor": light_blue})
            nodes.append({"id": "Interest Expense", "nodeColor": orange})
            nodes.append({"id": "Net Interest Income", "nodeColor": light_blue})
            if input["net_premium_income"] != 0:
                nodes.append({"id": "Net Premium Income", "nodeColor": light_blue})
            nodes.append({"id": "Non Interest Income", "nodeColor": light_blue})
            nodes.append({"id": "Total Revenue", "nodeColor": light_blue})

            if input["operating_income"] >= 0:
                nodes.append({"id": "Operating Income", "nodeColor": dark_blue})
            else:
                nodes.append({"id": "Operating Income", "nodeColor": red})

            nodes.append({"id": "Operating and Credit-Related Expenses", "nodeColor": orange})
            nodes.append({"id": "Operating Expense", "nodeColor": orange})
            nodes.append({"id": "Provision for Impairment", "nodeColor": orange})

            for oeb in input["operating_expense_breakdown"]:
                nodes.append({"id": oeb["category"], "nodeColor": orange})

        else:
            for rb in input["revenue_breakdown"]:
                nodes.append({"id": rb["category"], "nodeColor": light_blue})
            nodes.append({"id": "Total Revenue", "nodeColor": light_blue})

            nodes.append({"id": "Cost of Revenue", "nodeColor": orange})
            if input["gross_income"] >= 0:
                nodes.append({"id": "Gross Profit", "nodeColor": dark_blue})
            else:
                nodes.append({"id": "Gross Profit", "nodeColor": red})

            if input["operating_income"] >= 0:
                nodes.append({"id": "Operating Income", "nodeColor": dark_blue})
            else:
                nodes.append({"id": "Operating Income", "nodeColor": red})
            nodes.append({"id": "Operating Expense", "nodeColor": orange})

            for oeb in input["operating_expense_breakdown"]:
                nodes.append({"id": oeb["category"], "nodeColor": orange})

        return nodes

    output = {"nodes": make_nodes(), "links": make_links()}
    return output

def process_excel_file(file_path, financial_year, usd_idr_rate, bank_company):
    inputs,balance_sheets,cash_flows,employees, industry_breakdowns = generate_inputs(file_path, usd_idr_rate, bank_company)
    records = []

    for input in range(0,len(inputs)):
        errors = validate_input(inputs[input], bank_company)
        input_copy = inputs[input].copy()

        symbol = input_copy.pop('symbol')
        source_url = input_copy.pop('url')

        if len(errors) > 0:
            print(f"Errors for {symbol}: {errors}")
            output = None
        else:
            output = make_sankey_component(inputs[input], bank_company)

        source_url = source_url
        record = {
            'symbol': symbol + ".JK",
            'financial_year': financial_year,
            'sankey_component': output,
            'source_url': source_url,
            'income_stmt_metrics': input_copy,
            'balance_sheet_metrics': balance_sheets[input],
            'cash_flow_metrics': cash_flows[input],
            'employee_breakdown': employees[input],
            'industry_breakdown': industry_breakdowns[input],
            'updated_on': pd.to_datetime('now').strftime("%Y-%m-%d %H:%M:%S")
        }
        records.append(record)

    return inputs, records

## IDX Manual Input

### Run Data

In [ ]:
inputs, records = process_excel_file("Yet to Push FY2025 - Bank VI.xlsx", 2025, jisdor['2025'], True)

/tmp/ipykernel_6690/1088484218.py:104: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  income_stmt['value'] = income_stmt['value'].fillna(0)


Finish extracting manual input data of PNBS


/tmp/ipykernel_6690/1088484218.py:104: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  income_stmt['value'] = income_stmt['value'].fillna(0)


Finish extracting manual input data of NOBU


/tmp/ipykernel_6690/1088484218.py:104: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  income_stmt['value'] = income_stmt['value'].fillna(0)


...

/tmp/ipykernel_6690/1088484218.py:104: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  income_stmt['value'] = income_stmt['value'].fillna(0)


Finish extracting manual input data of DNAR


/tmp/ipykernel_6690/1088484218.py:104: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  income_stmt['value'] = income_stmt['value'].fillna(0)


Finish extracting manual input data of BTPN
Finish extracting manual input data of BSWD
Errors for BTPN: ['Interest Income must be equal to the sum of interest income breakdown']


/tmp/ipykernel_6690/1088484218.py:104: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  income_stmt['value'] = income_stmt['value'].fillna(0)


In [ ]:
inputs, records = process_excel_file("idx_manual_input Non Bank XII FY2024.xlsx", 2024, jisdor['2024'], False)

FileNotFoundError: [Errno 2] No such file or directory: 'idx_manual_input Non Bank XII FY2024.xlsx'

In [ ]:
# check data
records

[{'symbol': 'PNBS.JK',
  'financial_year': 2025,
  'sankey_component': {'nodes': [{'id': 'Musyarakah profit sharing income',
     'nodeColor': 'hsl(195, 53%, 79%)'},
    {'id': 'Mudharabah profit sharing income',
     'nodeColor': 'hsl(195, 53%, 79%)'},
    {'id': 'Ijarah lease income', 'nodeColor': 'hsl(195, 53%, 79%)'},
    {'id': 'Murabahah margin income', 'nodeColor': 'hsl(195, 53%, 79%)'},
    {'id': 'Marketable securities', 'nodeColor': 'hsl(195, 53%, 79%)'},
    {'id': 'Placement with Bank Indonesia and other banks',
     'nodeColor': 'hsl(195, 53%, 79%)'},
    {'id': 'Interest Income', 'nodeColor': 'hsl(195, 53%, 79%)'},
    {'id': 'Interest Expense', 'nodeColor': 'hsl(39, 100%, 50%)'},
    {'id': 'Net Interest Income', 'nodeColor': 'hsl(195, 53%, 79%)'},
    {'id': 'Non Interest Income', 'nodeColor': 'hsl(195, 53%, 79%)'},
    {'id': 'Total Revenue', 'nodeColor': 'hsl(195, 53%, 79%)'},
    {'id': 'Operating Income', 'nodeColor': 'hsl(240, 100%, 50%)'},
    {'id': 'Operating an

### Push to DB

In [ ]:
from google.colab import userdata

In [ ]:
url = userdata.get("SUPABASE_URL")
key = userdata.get("SUPABASE_SECRET_KEY")
supabase = create_client(url, key)

In [ ]:
supabase.table('idx_manual_input').upsert(records).execute()

APIResponse(data=[{'symbol': 'PNBS.JK', 'updated_on': '2026-04-29T05:28:33+00:00', 'financial_year': 2025, 'sankey_component': {'links': [{'value': 810981559000, 'source': 'Musyarakah profit sharing income', 'target': 'Interest Income'}, {'value': 115417389000, 'source': 'Mudharabah profit sharing income', 'target': 'Interest Income'}, {'value': 80216695000, 'source': 'Ijarah lease income', 'target': 'Interest Income'}, {'value': 4228794000, 'source': 'Murabahah margin income', 'target': 'Interest Income'}, {'value': 159944595000, 'source': 'Marketable securities', 'target': 'Interest Income'}, {'value': 31898642000, 'source': 'Placement with Bank Indonesia and other banks', 'target': 'Interest Income'}, {'value': 378402323000, 'source': 'Interest Income', 'target': 'Net Interest Income'}, {'value': 824285351000, 'source': 'Interest Income', 'target': 'Interest Expense'}, {'value': 378402323000, 'source': 'Net Interest Income', 'target': 'Total Revenue'}, {'value': 103210739000, 'sourc

## Company Customer Breakdown

In [ ]:
financial_year = 2025
jisdor[f'{financial_year}']

16785

In [ ]:
import numpy as np
import pandas as pd
from supabase import create_client
from postgrest import APIError
import json

def revenue_breakdown(file_name,financial_year):
    xl = pd.ExcelFile(file_name)

    list_sheet = xl.sheet_names  # see all sheet names

    revenue_data = pd.DataFrame()

    for i in list_sheet:
        df = pd.read_excel(file_name, sheet_name=i, header=None)

        data = df[[0,1]]
        data = data.rename(columns={0:'key', 1:'value'})
        data = data.dropna(subset=['key'])
        data = data.set_index('key')
        metadata = data.loc['symbol':'currency'].copy()
        currency = metadata.loc['currency'].value

        try:
            df = df.iloc[4:,9:12]
            df.columns = ["client_name","client_ticker","revenue_amount"]
            df = df[(~df.revenue_amount.isna()) & (df.client_name != "sum")]
            df['supplier_ticker'] = f"{i}.JK"
            df['financial_year'] = financial_year
            df = df[["supplier_ticker","client_name","client_ticker","financial_year",'revenue_amount']]

            if currency == "USD":
              df['revenue_amount'] = df['revenue_amount'] * jisdor[f'{financial_year}']
            revenue_data = pd.concat([revenue_data,df])
        except:
            print(f"No revenue breakdown for {i}")

    # remove non-number rows of revenue amount
    revenue_data = revenue_data[pd.to_numeric(revenue_data['revenue_amount'], errors='coerce').notnull()]
    # replace numpy's NaN at "client_ticker" column with None so that the postgrest can accept the value as NULL
    revenue_data = revenue_data.astype({"client_ticker": 'object'})
    revenue_data.loc[:, "client_ticker"] = revenue_data["client_ticker"].replace({np.nan: None})
    return revenue_data

def upsert_data(data, table: str):
    from google.colab import userdata
    url = userdata.get("SUPABASE_URL")
    key = userdata.get("SUPABASE_SECRET_KEY")
    supabase = create_client(url, key)

    # Delete data if want to update data
    uniq_supp = data.drop_duplicates(['supplier_ticker','financial_year']).reset_index()
    for i in range (0, uniq_supp.shape[0]):
        supabase.table(table).delete().eq('supplier_ticker',uniq_supp.loc[i,'supplier_ticker']).eq('financial_year',uniq_supp.loc[0,'financial_year']).execute()

    cleaned_data_list = []
    for i in range(0,data.shape[0]):
        # Assuming data is your DataFrame and iloc[0] selects the first row.
        data_dict = data.iloc[i].to_dict()
        # Convert numpy data types to native Python types
        cleaned_data_dict = {k: int(v) if isinstance(v, np.int64) else v for k, v in data_dict.items()}
        cleaned_data_list.append(cleaned_data_dict)

    try:
        # Insert the cleaned dictionary into the Supabase table
        # Assuming the datas are summarized correctly and ignoring updates, ignore existing rows
        supabase.table(table_name=table).upsert(
            cleaned_data_list,
            ignore_duplicates=True
        ).execute()
    except APIError as e:
        return f"Failed to update with error: {e}."

    return "Finish Upsert Data"

In [ ]:
cust_df = revenue_breakdown("Yet to Push FY2025 - Bank VI.xlsx",2025)

In [ ]:
upsert_data(cust_df, "idx_company_customer")

'Finish Upsert Data'

In [ ]:
import json

def find_non_compliant_floats(data):
    if isinstance(data, list):
        for item in data:
            find_non_compliant_floats(item)
    elif isinstance(data, dict):
        for key, value in data.items():
            if isinstance(value, float):
                if not (-1e300 < value < 1e300):  # Check if outside a reasonable range
                    print(f"Non-compliant float found: {value} at key: {key}")
            else:
                find_non_compliant_floats(value)

find_non_compliant_floats(records)

# SGX

In [ ]:
currency_exchange_rate = {
    'THB':{
        '2024-09-30': 0.039659,
        '2024-12-31': 0.039688,
        '2025-03-31': 0.039505,
        '2025-06-30': 0.039153
    },
    'MYR':{
        '2024-09-30': 0.3118,
        '2024-12-31': 0.3043,
        '2025-03-31': 0.3025,
        '2025-06-30': 0.3025
    },
    'USD':{
        '2024-09-30': 1.2806,
        '2024-12-31': 1.3603,
        '2025-03-31': 1.3410,
        '2025-06-30': 1.2758
    }
}

In [ ]:
def _get_sheet_names(excel_file_path):
    excel_file = pd.ExcelFile(excel_file_path)
    sheet_names = excel_file.sheet_names
    return sheet_names


def _filter_others_keys(row: pd.Series):
    """
    Removes keys containing \"Other\" by replacing their value with NaN,
    and return its values as series to be summed

    MODIFIES the DataFrame directly!
    """
    if re.search("^[Oo]ther", row.name):
        value = row.value
        row.value = np.NaN
        return value

def none_value_extractor(metrics):
    if pd.isna(metrics):
        return None
    else:
        return metrics

def _reduce_others_keys(df: pd.DataFrame):
    """
    Groups all \"Other\" breakdown entries into a single \"Others\" by summing them up

    MODIFIES the DataFrame directly!
    """

    others_rev = df[df.index.str.contains("^[Oo]ther")]
    df = df[~df.index.str.contains("^[Oo]ther")]

    if others_rev.shape[0] > 0:
        df.loc["Others", "value"] = int(others_rev["value"].sum())
        df.loc["Others", "category"] = others_rev['category'].iloc[0]
        return df
    else:
        return df

def _process_sheet(excel_file_path, sheet_name, curreny_exchange_rate, company):
    df = pd.read_excel(excel_file_path, sheet_name=sheet_name, header=None)

    date = df.iloc[1,4].strftime("%Y-%m-%d")

    data = df[[0,1]]
    data = data.rename(columns={0:'key', 1:'value'})
    data = data.dropna(subset=['key'])
    data = data.set_index('key')
    metadata = data.loc['symbol':'currency'].copy()
    # replace NaN value with None for postgrest json
    metadata = metadata.replace({np.nan: None})

    currency = metadata.loc['currency'].value

    if currency != "SGD":
        ex_rate = curreny_exchange_rate[currency][date]

    if company == "Bank":

        # Balance sheet
        bal_sheet = df.iloc[:,13:15]
        bal_sheet.columns = ["key",'value']
        bal_sheet['group'] = bal_sheet['key'].where(bal_sheet['value'].isna()).ffill()
        bal_sheet = bal_sheet.set_index("key")

        # Non-loan-asset breakdown
        non_loan_asset_bd = df[[15,16,17]]
        non_loan_asset_bd = non_loan_asset_bd.rename(columns={15:'class',16:'category', 17:'amount'}).dropna(subset=['category'])
        non_loan_asset_bd = non_loan_asset_bd.set_index('category')
        non_loan_asset_bd = non_loan_asset_bd.drop(index=["sum", "match",'Breakdown of: Non-loan asset']).reset_index()

        if currency != 'SGD':
            non_loan_asset_bd['amount'] = non_loan_asset_bd['amount'] * ex_rate

        balance_sheet = {
            "gross_loan": none_value_extractor(bal_sheet.loc['Gross Loan'].value),
            "allowance_for_loans": none_value_extractor(bal_sheet.loc['Allowance for Loans'].value),
            "net_loan": none_value_extractor(bal_sheet.loc['NET LOAN'].value),
            'earning_asset': none_value_extractor(non_loan_asset_bd[non_loan_asset_bd['class'] == "Earning Asset"].amount.sum() + bal_sheet.loc['NET LOAN'].value if pd.isna(bal_sheet.loc['Total Earning Asset'].value) else bal_sheet.loc['Total Earning Asset'].value),
            "non_loan_asset": none_value_extractor(bal_sheet.loc['Non-Loan Asset'].value),
            "total_asset": none_value_extractor(bal_sheet.loc['TOTAL ASSET'].value),
            "current_account": none_value_extractor(bal_sheet.loc['Current Account'].value),
            "savings_account": none_value_extractor(bal_sheet.loc['Savings Account'].value),
            "time_deposit": none_value_extractor(bal_sheet.loc['Time Deposits'].value),
            "total_deposit": none_value_extractor(bal_sheet.loc['TOTAL DEPOSIT'].value),
            "other_interest_bearing_liabilities": none_value_extractor(bal_sheet.loc['Other Interest-Bearing Liabilities'].value),
            "non_interest_bearing_liabilities": none_value_extractor(bal_sheet.loc['Non-Interest-Bearning Liabilities'].value),
            "total_liabilities": none_value_extractor(bal_sheet.loc['TOTAL LIABILITIES'].value),
            "total_equity": none_value_extractor(bal_sheet.loc["TOTAL SHAREHOLDER'S EQUITY"].value),
            "core_capital_tier1": none_value_extractor(bal_sheet.loc['Core Capital (Tier 1)'].value),
            "supplementary_capital_tier2": none_value_extractor(bal_sheet.loc['Supplementary Capital (Tier 2)'].value),
            "total_capital": none_value_extractor(bal_sheet.loc['TOTAL CAPITAL'].value),
            "credit_rwa": none_value_extractor(bal_sheet.loc['RWAs Considering Credit Risk'].value),
            "market_rwa": none_value_extractor(bal_sheet.loc['RWAs Considering Market Risk'].value),
            "operational_rwa": none_value_extractor(bal_sheet.loc['RWAs Considering Operational Risk'].value),
            "total_risk_weighted_asset": none_value_extractor(bal_sheet.loc['TOTAL RISK-WEIGHTED ASSETS'].value)
            }

        for metric in list(balance_sheet.keys()):
            try:
                balance_sheet[metric] = int(balance_sheet[metric])
                if currency != "SGD":
                    balance_sheet[metric] = int(balance_sheet[metric] * ex_rate)
            except:
                pass

        # Income Statement
        income_stmt = data.loc['interest income':'net income'].copy()
        income_stmt['value'] = income_stmt['value'].fillna(0)

        int_income_bd = df[[2,3,4]]
        int_income_bd = int_income_bd.rename(columns={2:'category',3:'key', 4:'value'}).dropna(subset=['key'])
        int_income_bd = int_income_bd.set_index('key')
        int_income_bd = int_income_bd.drop(index=["sum", "match",'Breakdown of: total revenue','Date'])
        int_income_bd = int_income_bd.dropna(subset=['value'])
        # Sum up all "Other" expenses into a single entry of "Others"
        int_income_bd = _reduce_others_keys(int_income_bd)
        int_income_bd = int_income_bd.reset_index()

        expense_bd = df[[5,6,7]]
        expense_bd = expense_bd.rename(columns={5:'category',6:'key', 7:'value'}).dropna(subset=['key'])
        expense_bd = expense_bd.set_index('key')
        expense_bd = expense_bd.drop(index=["sum", "match",'Breakdown of: operating expenses'])
        # check for positive expense; expense is standardized as having negative values only
        if (expense_bd['value'].values > 0).any():
            raise ValueError(f'Positive value detected in expense breakdown of sheet {sheet_name}')
        # Sum up all "Other" expenses into a single entry of "Others"
        expense_bd = _reduce_others_keys(expense_bd)
        # rename the expense index if any match is found in income breakdown
        for expense_idx in expense_bd.index:
            if expense_idx in int_income_bd['key'].values:
                expense_bd.rename(index={expense_idx: expense_idx + ' (expense)'}, inplace=True)
            elif expense_idx == "Others":
                expense_bd.rename(index={expense_idx: expense_idx + ' (expense)'}, inplace=True)

        expense_bd = expense_bd.reset_index()

        try:
            non_operating_income = income_stmt.loc['net non operating income/(expense)'].value
        except:
            non_operating_income = income_stmt.loc['net non operating income'].value

        input = {
            "symbol": metadata.loc['symbol'].value,
            "url": metadata.loc['url'].value,
            "interest_income": none_value_extractor(income_stmt.loc['interest income'].value),
            "interest_expense": none_value_extractor(-(income_stmt.loc['interest expenses'].value)),
            "net_interest_income": none_value_extractor(income_stmt.loc['net interest income'].value),
            "net_fee_and_commission_income": none_value_extractor(income_stmt.loc['net fee and commission income'].value),
            "net_trading_income": none_value_extractor(income_stmt.loc['net trading income'].value),
            "other_non_interest_income": none_value_extractor(income_stmt.loc['other non interest income'].value),
            "total_revenue": none_value_extractor(income_stmt.loc['total revenue'].value),
            "operating_expense": none_value_extractor(-(income_stmt.loc['operating expenses'].value)),
            "allowances_for_credit_and_other_losses": none_value_extractor(-(income_stmt.loc['allowances for credit and other losses'].value)),
            "amortization_of_intangible_assets": none_value_extractor(-(income_stmt.loc['amortization of intangible assets'].value)),
            "operating_income": none_value_extractor(income_stmt.loc['net operating income'].value),
            "non_operating_income_or_loss": none_value_extractor(non_operating_income),
            "pretax_income": none_value_extractor(income_stmt.loc['pretax income'].value),
            "income_taxes": none_value_extractor(-(income_stmt.loc['tax'].value)),
            "minorities": none_value_extractor(-(income_stmt.loc['minorities'].value)),
            "net_income": none_value_extractor(income_stmt.loc['net income'].value),
            "diluted_shares_outstanding": none_value_extractor(bal_sheet.loc['Weighted average shares outstanding'].value if bal_sheet.loc['Weighted average shares outstanding'].value != 0 else None),
            "int_income_breakdown": [{"class":int_income_bd.iloc[i,1],"category": int_income_bd.iloc[i,0], "amount": (int_income_bd.iloc[i,2])} for i in range (int_income_bd.shape[0])],
            "operating_expense_breakdown": [{"class":expense_bd.iloc[i,1],"category": expense_bd.iloc[i,0], "amount": int(-(expense_bd.iloc[i,2]))} for i in range (expense_bd.shape[0])]
            }

        for metric in ['interest_income', 'interest_expense', 'net_interest_income', 'net_fee_and_commission_income', 'net_trading_income', 'other_non_interest_income', 'total_revenue', 'operating_expense', 'allowances_for_credit_and_other_losses', 'amortization_of_intangible_assets', 'operating_income', 'non_operating_income_or_loss', 'pretax_income', 'income_taxes', 'minorities', 'net_income']:
            input[metric] = int(input[metric])
            if currency != "SGD":
                input[metric] = int(input[metric] * ex_rate)

        for breakdown in ['int_income_breakdown', 'operating_expense_breakdown']:
            for item in input[breakdown]:
                item['amount'] = int(item['amount'])
                if currency != "SGD":
                    item['amount'] = int(item['amount'] * ex_rate)

        # sector loan breakdown by economic sector
        sector_loan_breakdown = bal_sheet[bal_sheet.group == "Breakdown of: loan receivable (by economic sector)"]
        sector_loan_breakdown = sector_loan_breakdown.drop(index=["sum", 'Breakdown of: loan receivable (by economic sector)'])
        sector_loan_breakdown = sector_loan_breakdown.dropna(subset=['value'])
        sector_loan_breakdown =sector_loan_breakdown.reset_index()
        if currency != "SGD":
            sector_loan_breakdown['value'] = sector_loan_breakdown['value'] * ex_rate

        # customer breakdown
        customer_breakdown = df.iloc[5:,9:12]
        customer_breakdown.columns = ["client_name","client_ticker","revenue_amount"]
        customer_breakdown = customer_breakdown[(~customer_breakdown.revenue_amount.isna()) & (customer_breakdown.client_name != "sum")]
        if currency != "SGD":
            customer_breakdown['revenue_amount'] = customer_breakdown['revenue_amount'] * ex_rate
        customer_breakdown['revenue_amount'] = customer_breakdown['revenue_amount'].astype('int')

        if customer_breakdown.dropna(subset="client_ticker").shape[0] == 0:
            customer_breakdown = customer_breakdown[['client_name','revenue_amount']]

        industry_breakdown = {
            "loan_by_economic_sectors": dict(zip(sector_loan_breakdown["key"], sector_loan_breakdown["value"])),
            "non_loan_asset": non_loan_asset_bd.to_dict(orient='records'),
            "customer_breakdown": dict(zip(customer_breakdown["client_name"], customer_breakdown["revenue_amount"]))
        }

        # Cash Flow
        cash_fl = df[[19,20]]#.dropna()
        cash_fl.columns = ["key",'value']
        cash_fl = cash_fl.set_index("key")

        cash_flow = {
                    "operating_cash_flow": cash_fl.loc['Cash Flows from Operating Activities'].value,
                    "investing_cash_flow": cash_fl.loc['Cash Flows from Investing Activities'].value,
                    "financing_cash_flow": cash_fl.loc['Cash Flows from Financing Activities'].value,
                    "net_cash_flow": cash_fl.loc['NET INCREASE/DECREASED'].value,
                    "capital_expenditure": none_value_extractor(cash_fl.loc['CAPITAL EXPENDITURE'].value),
                    "free_cash_flow": none_value_extractor(cash_fl.loc['Cash Flows from Operating Activities'].value - cash_fl.loc['CAPITAL EXPENDITURE'].value),
                    }

        for metric in list(cash_flow.keys()):
            if currency != "SGD":
                try:
                    cash_flow[metric] = int(cash_flow[metric])
                    cash_flow[metric] = int(cash_flow[metric] * ex_rate)
                except:
                    pass

        # Employee
        employee = df.iloc[:,22:24]
        employee.columns = ['key','value']

        employee = employee.dropna(subset="key").iloc[1:,:].set_index("key")

        employee.index = ["permanent_employee","contract_employee","others_employee","total_employee"]

        if employee.loc['total_employee'].value == 0:
            employee = None
        else:
            employee.value = employee.value.astype('Int64')
            employee = employee["value"].to_dict()

    elif company == "REIT":
        # Income Statement Metrics
        income_stmt = data.loc['total revenue':'FFO'].copy()
        income_stmt['value'] = income_stmt['value'].fillna(0)

        revenue_bd = df[[2,3,4]]
        revenue_bd = revenue_bd.rename(columns={2:'category',3:'key', 4:'value'}).dropna(subset=['key'])
        revenue_bd = revenue_bd.set_index('key')
        revenue_bd = revenue_bd.drop(index=["sum", "match",'Breakdown of: total revenue','Date'])

        ## Sum up all "Other" expenses into a single entry of "Others"
        revenue_bd =_reduce_others_keys(revenue_bd)
        revenue_bd.reset_index(inplace=True)

        # Balance sheet Metrics
        bal_sheet = df.iloc[:50,13:15]
        bal_sheet.columns = ["key",'value']

        bal_sheet = bal_sheet.set_index("key")

        try:
            expense_bd = df[[5,6,7,8]]
            expense_bd = expense_bd.rename(columns={5:'category',6:'key', 7:'value', 8:'group'})

            expense_bd = expense_bd[~expense_bd.key.isin(["sum","match",'Breakdown of: operating expenses'])]

            expense_bd["group"] = expense_bd["group"].ffill()

            expense_bd = expense_bd.dropna()

            expense_bd["key"] = expense_bd["group"] + '-' + expense_bd["key"]

            expense_bd = expense_bd.drop(['group'], axis=1)
        except:
            expense_bd = df[[5,6,7]]
            expense_bd = expense_bd.rename(columns={5:'category',6:'key', 7:'value'}).dropna(subset=['key'])
            expense_bd = expense_bd.set_index('key')
            expense_bd = expense_bd.drop(index=["sum", "match",'Breakdown of: operating expenses'])
            # check for positive expense; expense is standardized as having negative values only
            if (expense_bd['value'].values > 0).any():
                raise ValueError(f'Positive value detected in expense breakdown of sheet {sheet_name}')
            # Sum up all "Other" expenses into a single entry of "Others"
            expense_bd = _reduce_others_keys(expense_bd)
            # rename the expense index if any match is found in income breakdown
            for expense_idx in expense_bd.index:
                if expense_idx in revenue_bd['key'].values:
                    expense_bd.rename(index={expense_idx: expense_idx + ' (expense)'}, inplace=True)
                elif expense_idx == "Others":
                    expense_bd.rename(index={expense_idx: expense_idx + ' (expense)'}, inplace=True)

            expense_bd.reset_index(inplace=True)

            expense_bd["value"] = expense_bd["value"].astype('int')

        input = {
            "symbol": metadata.loc['symbol'].value,
            "url": metadata.loc['url'].value,
            "total_revenue": none_value_extractor(income_stmt.loc['total revenue'].value),
            "cost_of_revenue": none_value_extractor(-(income_stmt.loc['cost of revenue'].value)),
            "gross_income": none_value_extractor(income_stmt.loc['gross income'].value),
            "operating_expense": none_value_extractor(-(income_stmt.loc['operating expenses'].value)),
            "operating_income": none_value_extractor(income_stmt.loc['net operating income'].value),
            "non_operating_income_or_loss": none_value_extractor(income_stmt.loc['net non operating income/(expenses)'].value),
            "pretax_income": none_value_extractor(income_stmt.loc['pretax income'].value),
            "income_taxes":  none_value_extractor(-(income_stmt.loc['tax'].value)),
            "net_income": none_value_extractor(income_stmt.loc['net income'].value),
            "minorities": none_value_extractor(income_stmt.loc['minorities'].value),
            "perpetual_security_holders": none_value_extractor(income_stmt.loc['perpetual security holders'].value),
            "unitholders": none_value_extractor(income_stmt.loc['unitholders'].value),
            "interest_expense_non_operating": none_value_extractor(-(income_stmt.loc['non operating interest expense'].value)),
            "ebit": none_value_extractor(income_stmt.loc['ebit'].value),
            "ebitda": none_value_extractor(income_stmt.loc['ebitda'].value if pd.isna(income_stmt.loc['ebitda'].value) else income_stmt.loc['ebit'].value + income_stmt.loc['depreciation and amortization'].value),
            "net_property_sales": none_value_extractor(income_stmt.loc['gain/(loss) on property sales'].value),
            "funds_from_operation": none_value_extractor(income_stmt.loc['FFO'].value),
            "diluted_shares_outstanding": none_value_extractor(bal_sheet.loc['Weighted average shares outstanding'].value),
            "revenue_breakdown": [{"class":revenue_bd.iloc[i,1],"category": revenue_bd.iloc[i,0], "amount": (revenue_bd.iloc[i,2])} for i in range (revenue_bd.shape[0])],
            "operating_expense_breakdown": [{"class":expense_bd.iloc[i,1],"category": expense_bd.iloc[i,0], "amount": int(-(expense_bd.iloc[i,2]))} for i in range (expense_bd.shape[0])]
            }

        balance_sheet = {
            "total_current_asset": none_value_extractor(bal_sheet.loc['Current Asset'].value),
            "total_non_current_asset": none_value_extractor(bal_sheet.loc['Non-Current Asset'].value),
            "total_asset": none_value_extractor(bal_sheet.loc['TOTAL ASSET'].value),
            "total_current_liabilities": none_value_extractor(bal_sheet.loc['Current Liabilities'].value),
            "total_non_current_liabilities": none_value_extractor(bal_sheet.loc['Non-Current Liabilities'].value),
            "total_liabilities": none_value_extractor(bal_sheet.loc['TOTAL LIABILITIES'].value),
            "total_equity": none_value_extractor(bal_sheet.loc["TOTAL SHAREHOLDER'S EQUITY"].value),
            "working_capital": none_value_extractor(bal_sheet.loc["Working Capital"].value)
            }

        try:
            capex = bal_sheet.loc['CAPITAL EXPENDITURE'].value
        except:
            try:
                capex = bal_sheet.loc['Net PP&E (current)'].value - bal_sheet.loc['Net PP&E (previous year)'].value + bal_sheet.loc['Depreciation expenses (current)'].value
            except:
                capex = None

        cash_flow = {
            "operating_cash_flow": none_value_extractor(bal_sheet.loc['Cash Flows from Operating Activities'].value),
            "investing_cash_flow": none_value_extractor(bal_sheet.loc['Cash Flows from Investing Activities'].value),
            "financing_cash_flow": none_value_extractor(bal_sheet.loc['Cash Flows from Financing Activities'].value),
            "net_cash_flow": none_value_extractor(bal_sheet.loc['NET INCREASE/DECREASED'].value),
            "capital_expenditure": none_value_extractor(capex),
            "free_cash_flow": none_value_extractor(bal_sheet.loc['Cash Flows from Operating Activities'].value - capex if capex != None else None),
            }

        for metric in list(balance_sheet.keys()):
            balance_sheet[metric] = int(balance_sheet[metric])
            if currency != "SGD":
                balance_sheet[metric] = int(balance_sheet[metric] * ex_rate)

        for metric in list(cash_flow.keys()):
            if currency != "SGD":
                try:
                    cash_flow[metric] = int(cash_flow[metric])
                    cash_flow[metric] = int(cash_flow[metric] * ex_rate)
                except:
                    pass

        for metric in ['total_revenue', 'cost_of_revenue', 'gross_income', 'operating_expense', 'operating_income', 'non_operating_income_or_loss', 'pretax_income', 'income_taxes', 'net_income', 'minorities', 'perpetual_security_holders', 'unitholders', 'interest_expense_non_operating', 'ebit', 'ebitda', 'net_property_sales', 'funds_from_operation']:
            input[metric] = int(input[metric])
            if currency != "SGD":
                input[metric] = int(input[metric] * ex_rate)

        for breakdown in ['revenue_breakdown', 'operating_expense_breakdown']:
            for item in input[breakdown]:
                item['amount'] = int(item['amount'])
                if currency != "SGD":
                    item['amount'] = int(item['amount'] * ex_rate)

        # Employee
        employee = df.iloc[:,24:26]
        employee.columns = ['key','value']

        employee = employee.dropna(subset="key").iloc[1:,:].set_index("key")

        employee.index = ["permanent_employee","contract_employee","others_employee","total_employee"]

        if employee.loc['total_employee'].value == 0:
            employee = None
        else:
            employee.value = employee.value.astype('Int64')
            employee = employee["value"].to_dict()

        # Industry parameter breakdown
        prop_port = df.iloc[5:,16:23].reset_index(drop=True)
        prop_port.columns = ['country','category','name','notes','valuation','gross_income','occupancy_rate']

        # top 20 property
        top_property = prop_port.iloc[:prop_port.index[prop_port["country"] == "Breakdown of: property portfolio"][0] - 1,:]
        top_property = top_property.dropna(how='all')
        top_property['occupancy_rate'] = (top_property['occupancy_rate'].astype('float')/100).round(2)
        top_property.rename(columns={"notes":"ownership_pct"}, inplace=True)
        top_property['ownership_pct'] = (top_property['ownership_pct'].astype('float')/100).round(2)

        if currency != 'SGD':
            top_property['valuation'] = top_property['valuation'] * ex_rate
            top_property['gross_income'] = top_property['gross_income'] * ex_rate

        # property breakdown by country
        property_port = prop_port.iloc[prop_port.index[prop_port["country"] == "Breakdown of: property portfolio"][0] + 2:,:]
        property_port['occupancy_rate'] = (property_port['occupancy_rate'].astype('float')/100).round(2)
        property_port = property_port[property_port.name != 'sum']
        property_port.drop(['name'],axis=1,inplace=True)
        property_port.dropna(how='all', inplace= True)
        property_port.rename(columns={"notes":"property_counts"}, inplace=True)

        if currency != 'SGD':
            property_port['valuation'] = property_port['valuation'] * ex_rate
            property_port['gross_income'] = property_port['gross_income'] * ex_rate


        property_port_result = {}
        for _, row in property_port.iterrows():
            country = row["country"]
            category = row["category"]
            count = row["property_counts"]
            income = row["gross_income"]
            valuation = row["valuation"]
            property_port_result.setdefault(country, {})[category] = [count,income,valuation]

        # customer breakdown
        customer_breakdown = df.iloc[5:,9:12]
        customer_breakdown.columns = ["client_name","industry","revenue_pct"]
        customer_breakdown.reset_index(inplace=True, drop=True)

        cust_sector = customer_breakdown.iloc[customer_breakdown.index[customer_breakdown["client_name"] == "Breakdown of: by Gross Rental Income (GRI%)"][0] - 1:,:]
        cust_sector = cust_sector[(~cust_sector.revenue_pct.isna()) & (cust_sector.client_name != "sum")]
        cust_sector['revenue_pct'] = ((cust_sector['revenue_pct']).astype('float')/100).round(2)
        if cust_sector.dropna(subset='industry').shape[0] == 0:
            cust_sector = cust_sector[['client_name','revenue_pct']]
            cust_sector.columns = ['sector','revenue_pct']

        top_10_cust = customer_breakdown.iloc[:customer_breakdown.index[customer_breakdown["client_name"] == "Breakdown of: by Gross Rental Income (GRI%)"][0] - 1,:]
        top_10_cust = top_10_cust[(~top_10_cust.revenue_pct.isna()) & (top_10_cust.client_name != "sum")]
        top_10_cust['revenue_pct'] = ((top_10_cust['revenue_pct']).astype('float')/100).round(2)
        if top_10_cust.dropna(subset='industry').shape[0] == 0:
            top_10_cust = top_10_cust[['client_name','revenue_pct']]

        industry_breakdown = {
            "top_10_gri%_customers": top_10_cust.reset_index(drop=True).to_dict(orient='records'),
            "gross_rental_income_by_sectors": dict(zip(cust_sector["sector"], cust_sector["revenue_pct"])),
            "property_portfolio_top_20": top_property.apply(lambda row: row.dropna().to_dict(), axis=1).tolist(),
            "property_counts_by_country": property_port_result
        }

    else:  # NOT bank company and reit
        # Income Statement Metrics
        income_stmt = data.loc['total revenue':'ebitda'].copy()
        income_stmt['value'] = income_stmt['value'].fillna(0)

        revenue_bd = df[[2,3,4]]
        revenue_bd = revenue_bd.rename(columns={2:'category',3:'key', 4:'value'}).dropna(subset=['key'])
        revenue_bd = revenue_bd.set_index('key')
        revenue_bd = revenue_bd.drop(index=["sum", "match",'Breakdown of: total revenue','Date'])

        ## Sum up all "Other" expenses into a single entry of "Others"
        revenue_bd =_reduce_others_keys(revenue_bd)
        revenue_bd.reset_index(inplace=True)

        # Balance sheet Metrics
        bal_sheet = df.iloc[:50,13:15]
        bal_sheet.columns = ["key",'value']

        bal_sheet = bal_sheet.set_index("key")

        try:
            expense_bd = df[[5,6,7,8]]
            expense_bd = expense_bd.rename(columns={5:'category',6:'key', 7:'value', 8:'group'})

            expense_bd = expense_bd[~expense_bd.key.isin(["sum","match",'Breakdown of: operating expenses'])]

            expense_bd["group"] = expense_bd["group"].ffill()

            expense_bd = expense_bd.dropna()

            expense_bd["key"] = expense_bd["group"] + '-' + expense_bd["key"]

            expense_bd = expense_bd.drop(['group'], axis=1)
        except:
            expense_bd = df[[5,6,7]]
            expense_bd = expense_bd.rename(columns={5:'category',6:'key', 7:'value'}).dropna(subset=['key'])
            expense_bd = expense_bd.set_index('key')
            expense_bd = expense_bd.drop(index=["sum", "match",'Breakdown of: operating expenses'])
            # check for positive expense; expense is standardized as having negative values only
            if (expense_bd['value'].values > 0).any():
                raise ValueError(f'Positive value detected in expense breakdown of sheet {sheet_name}')
            # Sum up all "Other" expenses into a single entry of "Others"
            expense_bd = _reduce_others_keys(expense_bd)
            # rename the expense index if any match is found in income breakdown
            for expense_idx in expense_bd.index:
                if expense_idx in revenue_bd['key'].values:
                    expense_bd.rename(index={expense_idx: expense_idx + ' (expense)'}, inplace=True)
                elif expense_idx == "Others":
                    expense_bd.rename(index={expense_idx: expense_idx + ' (expense)'}, inplace=True)

            expense_bd.reset_index(inplace=True)

            expense_bd["value"] = expense_bd["value"].astype('int')

        input = {
            "symbol": metadata.loc['symbol'].value,
            "url": metadata.loc['url'].value,
            "total_revenue": none_value_extractor(income_stmt.loc['total revenue'].value),
            "cost_of_revenue": none_value_extractor(-(income_stmt.loc['cost of revenue'].value)),
            "gross_income": none_value_extractor(income_stmt.loc['gross income'].value),
            "operating_expense": none_value_extractor(-(income_stmt.loc['operating expenses'].value)),
            "operating_income": none_value_extractor(income_stmt.loc['net operating income'].value),
            "non_operating_income_or_loss": none_value_extractor(income_stmt.loc['net non operating income/(expenses)'].value),
            "pretax_income": none_value_extractor(income_stmt.loc['pretax income'].value),
            "income_taxes":  none_value_extractor(-(income_stmt.loc['tax'].value)),
            "net_income": none_value_extractor(income_stmt.loc['net income'].value),
            "interest_expense_non_operating": none_value_extractor(-(income_stmt.loc['non operating interest expense'].value)),
            "ebit": none_value_extractor(income_stmt.loc['ebit'].value),
            "ebitda": none_value_extractor(income_stmt.loc['ebitda'].value if pd.isna(income_stmt.loc['ebitda'].value) else income_stmt.loc['ebit'].value + income_stmt.loc['depreciation and amortization'].value),
            "diluted_shares_outstanding": none_value_extractor(bal_sheet.loc['Weighted average shares outstanding'].value),
            "revenue_breakdown": [{"class":revenue_bd.iloc[i,1],"category": revenue_bd.iloc[i,0], "amount": (revenue_bd.iloc[i,2])} for i in range (revenue_bd.shape[0])],
            "operating_expense_breakdown": [{"class":expense_bd.iloc[i,1],"category": expense_bd.iloc[i,0], "amount": int(-(expense_bd.iloc[i,2]))} for i in range (expense_bd.shape[0])]
            }

        balance_sheet = {
            "total_current_asset": none_value_extractor(bal_sheet.loc['Current Asset'].value),
            "total_non_current_asset": none_value_extractor(bal_sheet.loc['Non-Current Asset'].value),
            "total_asset": none_value_extractor(bal_sheet.loc['TOTAL ASSET'].value),
            "total_current_liabilities": none_value_extractor(bal_sheet.loc['Current Liabilities'].value),
            "total_non_current_liabilities": none_value_extractor(bal_sheet.loc['Non-Current Liabilities'].value),
            "total_liabilities": none_value_extractor(bal_sheet.loc['TOTAL LIABILITIES'].value),
            "total_equity": none_value_extractor(bal_sheet.loc["TOTAL SHAREHOLDER'S EQUITY"].value),
            "working_capital": none_value_extractor(bal_sheet.loc["Working Capital"].value)
            }

        try:
            capex = bal_sheet.loc['CAPITAL EXPENDITURE'].value
        except:
            try:
                capex = bal_sheet.loc['Net PP&E (current)'].value - bal_sheet.loc['Net PP&E (previous year)'].value + bal_sheet.loc['Depreciation expenses (current)'].value
            except:
                capex = None

        cash_flow = {
            "operating_cash_flow": none_value_extractor(bal_sheet.loc['Cash Flows from Operating Activities'].value),
            "investing_cash_flow": none_value_extractor(bal_sheet.loc['Cash Flows from Investing Activities'].value),
            "financing_cash_flow": none_value_extractor(bal_sheet.loc['Cash Flows from Financing Activities'].value),
            "net_cash_flow": none_value_extractor(bal_sheet.loc['NET INCREASE/DECREASED'].value),
            "capital_expenditure": none_value_extractor(capex),
            "free_cash_flow": none_value_extractor(bal_sheet.loc['Cash Flows from Operating Activities'].value - capex if capex != None else None),
            }

        for metric in list(balance_sheet.keys()):
            balance_sheet[metric] = int(balance_sheet[metric])
            if currency != "SGD":
                balance_sheet[metric] = int(balance_sheet[metric] * ex_rate)

        for metric in list(cash_flow.keys()):
            if currency != "SGD":
                try:
                    cash_flow[metric] = int(cash_flow[metric])
                    cash_flow[metric] = int(cash_flow[metric] * ex_rate)
                except:
                    pass

        for metric in ['total_revenue', 'cost_of_revenue', 'gross_income', 'operating_expense', 'operating_income', "non_operating_income_or_loss", 'pretax_income', 'income_taxes', 'net_income', 'interest_expense_non_operating', 'ebit', 'ebitda']:
            input[metric] = int(input[metric])
            if currency != "SGD":
                input[metric] = int(input[metric] * ex_rate)

        for breakdown in ['revenue_breakdown', 'operating_expense_breakdown']:
            for item in input[breakdown]:
                item['amount'] = int(item['amount'])
                if currency != "SGD":
                    item['amount'] = int(item['amount'] * ex_rate)

        # Employee
        try:
            employee = df.iloc[:,16:18]
            employee.columns = ['key','value']

            employee = employee.dropna(subset="key").iloc[1:,:].set_index("key")

            employee.index = ["permanent_employee","contract_employee","others_employee","total_employee"]

            if employee.loc['total_employee'].value == 0:
                employee = None
            else:
                employee.value = employee.value.astype('Int64')
                employee = employee["value"].to_dict()
        except:
            employee = None

       # Industry parameter breakdown
        breakdown = df.iloc[49:,13:15]
        breakdown.columns = ['key','value']
        breakdown.dropna(subset="value", inplace=True)
        breakdown = breakdown[breakdown.value != 0]

        # Customer Breakdown
        customer = df.iloc[:,9:12]
        customer = customer.dropna(how='all')
        customer.columns = ['client_name','client_ticker','revenue_amount']
        customer = customer[~customer['client_name'].isin(['sum','Breakdown of: customers'])]
        if currency != 'SGD':
            customer['revenue_amount'] = customer['revenue_amount'] * ex_rate

        # Industry Breakdown
        industry_breakdown = {
            'key_performance_index': dict(zip(breakdown["key"], breakdown["value"])),
            'customer breakdown': customer.apply(lambda row: row.dropna().to_dict(), axis=1).tolist()
            # breakdown.to_dict(orient='records')
        }

        industry_breakdown = {k: v for k, v in industry_breakdown.items() if v != {}}

    return input,balance_sheet,cash_flow, employee, industry_breakdown,date

def generate_inputs(excel_file_path, usd_sgd_rate, company):
    inputs = []
    balance_sheets = []
    cash_flows = []
    employees = []
    industry_breakdowns = []
    dates = []
    sheet_names = _get_sheet_names(excel_file_path)
    for sheet_name in sheet_names:
        input, balance_sheet, cash_flow, employee, industry_breakdown, date = _process_sheet(excel_file_path, sheet_name, usd_sgd_rate, company)
        inputs.append(input)
        balance_sheets.append(balance_sheet)
        cash_flows.append(cash_flow)
        employees.append(employee)
        dates.append(date)
        industry_breakdowns.append(industry_breakdown)

        print(f'Finish extracting manual input data of {sheet_name}')
    return inputs,balance_sheets,cash_flows, employees, industry_breakdowns,dates

def make_sankey_component(input, company):
    def make_links():
        links = []

        if company == "Bank":
            for iib in input["int_income_breakdown"]:
                links.append(
                    {
                        "source": iib["category"],
                        "target": "Interest Income",
                        "value": iib["amount"],
                    }
                )

            links.append(
                {
                    "source": "Interest Income",
                    "target": "Net Interest Income",
                    "value": input["net_interest_income"],
                }
            )

            links.append(
                {
                    "source": "Interest Income",
                    "target": "Interest Expense",
                    "value": input["interest_expense"],
                }
            )

            links.append(
                {
                    "source": "Net Interest Income",
                    "target": "Total Revenue",
                    "value": input["net_interest_income"],
                }
            )

            links.append(
                {
                    "source": "Net Fee and Commission Income",
                    "target": "Total Revenue",
                    "value": input["net_fee_and_commission_income"],
                }
            )
            links.append(
                {
                    "source": "Net Trading Income",
                    "target": "Total Revenue",
                    "value": input["net_trading_income"],
                }
            )

            links.append(
                {
                    "source": "Other Non Interest Income",
                    "target": "Total Revenue",
                    "value": input["other_non_interest_income"],
                }
            )

            if input["operating_income"] >= 0:
                links.append(
                    {
                        "source": "Total Revenue",
                        "target": "Operating Income",
                        "value": input["operating_income"],
                    }
                )
                links.append(
                    {
                        "source": "Total Revenue",
                        "target": "Operating and Credit-Related Expenses",
                        "value": input["operating_expense"] + input["allowances_for_credit_and_other_losses"] + input["amortization_of_intangible_assets"],
                    }
                )
            else:
                links.append(
                    {
                        "source": "Operating Income",
                        "target": "Operating and Credit-Related Expenses",
                        "value": -(input["operating_income"]),
                    }
                )
                links.append(
                    {
                        "source": "Total Revenue",
                        "target": "Operating and Credit-Related Expenses",
                        "value": input["total_revenue"],
                    }
                )

            links.append(
                {
                    "source": "Operating and Credit-Related Expenses",
                    "target": "Operating Expense",
                    "value": input["operating_expense"],
                }
            )

            links.append(
                {
                    "source": "Operating and Credit-Related Expenses",
                    "target": "Allowances for Credit and Other Losses",
                    "value": input["allowances_for_credit_and_other_losses"],
                }
            )

            links.append(
                {
                    "source": "Operating and Credit-Related Expenses",
                    "target": "Amortization of Intangible Assets",
                    "value": input["amortization_of_intangible_assets"],
                }
            )

            for oeb in input["operating_expense_breakdown"]:
                links.append(
                    {
                        "source": "Operating Expense",
                        "target": oeb["category"],
                        "value": oeb["amount"],
                    }
        )

        else:
            for rb in input["revenue_breakdown"]:
                links.append(
                    {
                        "source": rb["category"],
                        "target": "Total Revenue",
                        "value": rb["amount"],
                    }
                )

            if input["gross_income"] >= 0:
                links.append(
                    {
                        "source": "Total Revenue",
                        "target": "Cost of Revenue",
                        "value": input["cost_of_revenue"],
                    }
                )

                links.append(
                    {
                        "source": "Total Revenue",
                        "target": "Gross Profit",
                        "value": input["gross_income"],
                    }
                )
                if input["operating_income"] >= 0:
                    links.append(
                        {
                            "source": "Gross Profit",
                            "target": "Operating Income",
                            "value": input["operating_income"],
                        }
                    )
                    links.append(
                        {
                            "source": "Gross Profit",
                            "target": "Operating Expense",
                            "value": input["operating_expense"],
                        }
                    )
                else:
                    links.append(
                        {
                            "source": "Operating Income",
                            "target": "Operating Expense",
                            "value": -(input["operating_income"]),
                        }
                    )
                    links.append(
                        {
                            "source": "Gross Profit",
                            "target": "Operating Expense",
                            "value": input["gross_income"],
                        }
                    )

            else:
                links.append(
                    {
                        "source": "Total Revenue",
                        "target": "Cost of Revenue",
                        "value": input["total_revenue"],
                    }
                )
                links.append(
                    {
                        "source": "Gross Profit",
                        "target": "Cost of Revenue",
                        "value": -(input["gross_income"]),
                    }
                )
                links.append(
                    {
                        "source": "Operating Income",
                        "target": "Gross Profit",
                        "value": -(input["gross_income"]),
                    }
                )
                links.append(
                    {
                        "source": "Operating Income",
                        "target": "Operating Expense",
                        "value": input["operating_expense"],
                    }
                )

            for oeb in input["operating_expense_breakdown"]:
                links.append(
                    {
                        "source": "Operating Expense",
                        "target": oeb["category"],
                        "value": oeb["amount"],
                    }
                )

        return links

    def make_nodes():
        red = "hsl(0, 100%, 50%)"
        orange = "hsl(39, 100%, 50%)"
        light_blue = "hsl(195, 53%, 79%)"
        dark_blue = "hsl(240, 100%, 50%)"

        nodes = []

        if company == "Bank":
            for iib in input["int_income_breakdown"]:
                nodes.append({"id": iib["category"], "nodeColor": light_blue})
            nodes.append({"id": "Interest Income", "nodeColor": light_blue})
            nodes.append({"id": "Interest Expense", "nodeColor": orange})
            nodes.append({"id": "Net Interest Income", "nodeColor": light_blue})
            nodes.append({"id": "Net Fee and Commission Income", "nodeColor": light_blue})
            nodes.append({"id": "Other Non Interest Income", "nodeColor": light_blue})
            nodes.append({"id": "Net Trading Income", "nodeColor": light_blue})
            nodes.append({"id": "Total Revenue", "nodeColor": light_blue})

            if input["operating_income"] >= 0:
                nodes.append({"id": "Operating Income", "nodeColor": dark_blue})
            else:
                nodes.append({"id": "Operating Income", "nodeColor": red})

            nodes.append({"id": "Operating and Credit-Related Expenses", "nodeColor": orange})
            nodes.append({"id": "Operating Expense", "nodeColor": orange})
            nodes.append({"id": "Amortization of Intangible Assets", "nodeColor": orange})
            nodes.append({"id": "Allowances for Credit and Other Losses", "nodeColor": orange})

            for oeb in input["operating_expense_breakdown"]:
                nodes.append({"id": oeb["category"], "nodeColor": orange})

        else:
            for rb in input["revenue_breakdown"]:
                nodes.append({"id": rb["category"], "nodeColor": light_blue})
            nodes.append({"id": "Total Revenue", "nodeColor": light_blue})

            nodes.append({"id": "Cost of Revenue", "nodeColor": orange})
            if input["gross_income"] >= 0:
                nodes.append({"id": "Gross Profit", "nodeColor": dark_blue})
            else:
                nodes.append({"id": "Gross Profit", "nodeColor": red})

            if input["operating_income"] >= 0:
                nodes.append({"id": "Operating Income", "nodeColor": dark_blue})
            else:
                nodes.append({"id": "Operating Income", "nodeColor": red})
            nodes.append({"id": "Operating Expense", "nodeColor": orange})

            for oeb in input["operating_expense_breakdown"]:
                nodes.append({"id": oeb["category"], "nodeColor": orange})

        return nodes

    output = {"nodes": make_nodes(), "links": make_links()}
    return output

def process_excel_file(file_path, usd_sgd_rate, company):
    inputs ,balance_sheets, cash_flows, employees, industry_breakdowns, dates = generate_inputs(file_path, usd_sgd_rate, company)
    records = []

    for input in range(0,len(inputs)):
        input_copy = inputs[input].copy()

        symbol = input_copy.pop('symbol')
        source_url = input_copy.pop('url')

        output = make_sankey_component(inputs[input], company)

        source_url = source_url
        record = {
            'symbol': symbol,
            'date': dates[input],
            'sankey_component': output,
            'financial_year':pd.to_datetime(dates[input]).year,
            'source_url': source_url,
            'income_stmt_metrics': input_copy,
            'balance_sheet_metrics': balance_sheets[input],
            'cash_flow_metrics': cash_flows[input],
            'employee_breakdown': none_value_extractor(employees[input]),
            'industry_breakdown': industry_breakdowns[input],
            'updated_on': pd.to_datetime('now').strftime("%Y-%m-%d %H:%M:%S")
        }
        records.append(record)

    return inputs, records

## Non bank and REIT Companies

In [ ]:
inputs, records = process_excel_file("SGX - FY 2024 - Others.xlsx", currency_exchange_rate, "Other")

## Bank Companies

In [ ]:
inputs, records = process_excel_file("SGX - FY2024 - Bank.xlsx", currency_exchange_rate, "Bank")

## REIT Companies

In [ ]:
inputs, records = process_excel_file("/Users/geraldbryan/Downloads/RERUN.xlsx", currency_exchange_rate, "REIT")

## Upload Data to DB

In [ ]:
def sanitize_for_json(obj):
    """
    Recursively make a Python object safe for JSON/JSONB:
    - None -> None (becomes null in JSON)
    - NaN, Inf, -Inf -> None
    - numpy types -> converted to native Python
    """
    if isinstance(obj, dict):
        return {k: sanitize_for_json(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [sanitize_for_json(v) for v in obj]
    elif isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return None
        return obj
    elif isinstance(obj, (np.generic,)):  # numpy scalar types
        return sanitize_for_json(obj.item())
    else:
        return obj


In [ ]:
# Upload to Database
supabase.table('sgx_manual_input').upsert(sanitize_for_json(records)).execute()